# 🏆 RSNA Knee Abnormality Detection — Full 5-Fold ConvNeXt-Tiny 2.5D Champion Pipeline
### 🚀 SOTA Champion Architecture: Channel-Gated FPN (Zero-Init), 3D Intra-Plane Volumetric Depth Block & Linear Focal Gating
---
**Key Pipeline Enhancements**:
1. **ConvNeXt-Tiny 2.5D Multi-Slice Backbone**: 21 slices (7 Sagittal, 7 Coronal, 7 Axial) $\times$ 3 adjacent channels (2.5D triplets).
2. **2D Channel-Gated Feature Pyramid Network (FPN)**: Stage 3 ($14\times 14$) + Stage 4 ($7\times 7$) LayerNorm-stabilized Top-Down 2D Fusion with channel-wise $\gamma \in \mathbb{R}^{1 \times 768}$ initialized at $0.0$.
3. **3D Intra-Plane Volumetric Depth Block**: 1D Depthwise Conv3 within each isolated imaging plane (zero cross-plane leakage, zero-init residual $\gamma=0.0$).
4. **Trainable GeM Spatial Pooling**: Power $p \in [1.0, 10.0]$, initialized at $p=3.0$, end-to-end differentiable (Zero Weight Decay).
5. **Plane-Decoupled Hierarchical Linear Focal Gating**:
   - Level 1: Intra-Plane Attention Linear Gating (Contextual + Broad Smoothing $\tau=2.0$ blended via zero-init linear `focal_gate`).
   - Level 2: Inter-Plane Gating with Gaussian anatomical priors, routing attention to relevant planes.
6. **Warm-Start Residual Anatomical Gaussian Slot Prior**: Clinically anchored $(\mu_0, \sigma_0, \pi_0)$ with bounded residual drift and adaptive lesion spread.
7. **Data Augmentation**: GPU Batch-level Affine + MRI Physics Gamma Transform + Gold-Preserving Mixup (`is_gold + is_gold[index]`).
8. **Class-Calibrated ASL + Soft-Margin Rank Loss**: Protected Asymmetric Loss base + continuous FN scaling + rank margin loss ($w=0.15$).
9. **Full 5-Fold Ensemble & SWA + Fold-Wise Percentile Rank Normalization**: Stochastic Weight Averaging over Top-3 epochs per fold, generating comprehensive Out-Of-Fold (OOF) predictions (`oof_predictions_kaggle_5folds.csv`).


In [1]:
# ── 1. Kaggle Environment & Dependencies ────────────────────────────────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!pip install -q timm pydicom transformers huggingface_hub iterative-stratification
print('✅ Dependencies installed successfully.')


✅ Dependencies installed successfully.


In [2]:
import os
import glob
import timm

# ── 1. Hardcoded Direct Kaggle Input Paths (Zero Search Overhead) ───────────
TRAIN_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
if not os.path.exists(TRAIN_CSV) and os.path.exists("/kaggle/input/rsna-knee-abnormality-detection/train.csv"):
    TRAIN_CSV = "/kaggle/input/rsna-knee-abnormality-detection/train.csv"

NPY_PATH = "/kaggle/input/datasets/mediumguy/rsna-preprocessing-data/knee_train_21slices_224.npy"
if not os.path.exists(NPY_PATH) and os.path.exists("/kaggle/input/rsna-preprocessing-data/knee_train_21slices_224.npy"):
    NPY_PATH = "/kaggle/input/rsna-preprocessing-data/knee_train_21slices_224.npy"

LLM_CSV = "/kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv"
if not os.path.exists(LLM_CSV):
    for cand_pat in [
        "/kaggle/input/rsna-llm-dataset/llm_labels_v4_blend.csv",
        "/kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv",
        "/kaggle/input/**/llm_labels_v4_blend.csv",
        "./llm_labels_v4_blend.csv",
        "/content/rsna-llm-dataset/llm_labels_v4_blend.csv",
        "/kaggle/input/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv"
    ]:
        matches = glob.glob(cand_pat, recursive=True)
        if matches and os.path.exists(matches[0]):
            LLM_CSV = matches[0]
            break

META_CSV = "/kaggle/input/datasets/mediumguy/rsna-preprocessing-data/train_metadata_21slices.csv"
if not os.path.exists(META_CSV) and os.path.exists("/kaggle/input/rsna-preprocessing-data/train_metadata_21slices.csv"):
    META_CSV = "/kaggle/input/rsna-preprocessing-data/train_metadata_21slices.csv"

# ── 2. Direct ConvNeXt Backbone Download via timm Library ───────────────────
MODEL_PATH = "convnext_tiny.in12k_ft_in1k"
print("⬇️ Pre-downloading & caching ConvNeXt-Tiny weights via timm library (Internet: On)...")
_ = timm.create_model(MODEL_PATH, pretrained=True, num_classes=0)
print("   ✅ ConvNeXt-Tiny weights successfully downloaded and cached in PyTorch hub!")

# ── 3. High-Speed Local NVMe /tmp Caching on Kaggle (>2.5 GB/s) ─────────────
TMP_NPY = "/tmp/knee_train_21slices_224.npy"
if os.path.exists(NPY_PATH):
    if not os.path.exists(TMP_NPY):
        print(f"⚡ Fast copying Master Array to local NVMe SSD ({TMP_NPY}) for zero I/O wait (~5s)...")
        os.system(f"cp '{NPY_PATH}' '{TMP_NPY}'")
    if os.path.exists(TMP_NPY):
        NPY_PATH = TMP_NPY
        print(f"   ✅ Using local NVMe cached array: {TMP_NPY}")

print("\n🔍 Kaggle Input Status:")
print(f"   • Train CSV:         {TRAIN_CSV} (Exists: {os.path.exists(TRAIN_CSV)})")
print(f"   • LLM Labels CSV:    {LLM_CSV} (Exists: {os.path.exists(LLM_CSV)})")
print(f"   • Preprocessed NPY:  {NPY_PATH} (Exists: {os.path.exists(NPY_PATH)})")
print(f"   • Metadata CSV:      {META_CSV} (Exists: {os.path.exists(META_CSV)})")
print(f"   • ConvNeXt Backbone: {MODEL_PATH} (Direct timm download: Ready)")

⬇️ Pre-downloading & caching ConvNeXt-Tiny weights via timm library (Internet: On)...


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

   ✅ ConvNeXt-Tiny weights successfully downloaded and cached in PyTorch hub!
⚡ Fast copying Master Array to local NVMe SSD (/tmp/knee_train_21slices_224.npy) for zero I/O wait (~5s)...
   ✅ Using local NVMe cached array: /tmp/knee_train_21slices_224.npy

🔍 Kaggle Input Status:
   • Train CSV:         /kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv (Exists: True)
   • LLM Labels CSV:    /kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv (Exists: True)
   • Preprocessed NPY:  /tmp/knee_train_21slices_224.npy (Exists: True)
   • Metadata CSV:      /kaggle/input/datasets/mediumguy/rsna-preprocessing-data/train_metadata_21slices.csv (Exists: True)
   • ConvNeXt Backbone: convnext_tiny.in12k_ft_in1k (Direct timm download: Ready)


In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc, copy, random, warnings, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from sklearn.metrics import roc_auc_score, confusion_matrix
from tqdm.auto import tqdm
import timm
torch.backends.cudnn.benchmark = True  # ⚡ Accelerates ConvNeXt FP16 CuDNN kernel selection for fixed 224x224

warnings.filterwarnings("ignore")

# ── Dynamic Path Discovery for Kaggle & Local ────────────────────────────────
if "TRAIN_CSV" not in globals():
    TRAIN_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv" if os.path.exists("/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv") else ("./train.csv" if os.path.exists("./train.csv") else "/content/train.csv")
if "LLM_CSV" not in globals():
    LLM_CSV   = "/kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv" if os.path.exists("/kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv") else ("./llm_labels_v4_blend.csv" if os.path.exists("./llm_labels_v4_blend.csv") else "/content/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv")
if "NPY_PATH" not in globals():
    NPY_PATH  = "/kaggle/input/datasets/mediumguy/rsna-preprocessing-data/knee_train_21slices_224.npy" if os.path.exists("/kaggle/input/datasets/mediumguy/rsna-preprocessing-data/knee_train_21slices_224.npy") else ("./knee_train_21slices_224.npy" if os.path.exists("./knee_train_21slices_224.npy") else "/content/rsna-preprocessing-data/knee_train_21slices_224.npy")
if "META_CSV" not in globals():
    META_CSV  = "/kaggle/input/datasets/mediumguy/rsna-preprocessing-data/train_metadata_21slices.csv" if os.path.exists("/kaggle/input/datasets/mediumguy/rsna-preprocessing-data/train_metadata_21slices.csv") else ("./train_metadata_21slices.csv" if os.path.exists("./train_metadata_21slices.csv") else "/content/rsna-preprocessing-data/train_metadata_21slices.csv")
if "MODEL_PATH" not in globals():
    MODEL_PATH = "convnext_tiny.in12k_ft_in1k"

class CFG:
    seed                    = 42
    img_size                = 224          # ⚡ Native 224x224 resolution matching V3 preprocessed dataset
    num_slices              = 21           # 21 Slices: 7 Sagittal + 7 Coronal + 7 Axial
    num_slices_per_plane    = 7            # 7 Slices per plane (7 Sagittal + 7 Coronal + 7 Axial)
    in_chans                = 3            # 2.5D Triplets
    batch_size              = 16           # Batch size: 16 studies
    # accum_steps synchronized with Kaggle (Direct update per batch, N updates/epoch)
    backbone_chunk_size     = 672          # ⚡ 672 slices (eliminates forward split/cat overhead for 16*21=336 images)
    val_batch_size          = 64           # ⚡ Boosted validation batch size (faster eval without gradient memory overhead)
    num_workers             = 2            # ⚡ 2 Workers for stable, smooth parallel prefetching on Kaggle
    pin_memory              = True         # ⚡ Fast host-to-device memory transfers on Kaggle 30 GB RAM
    prefetch_factor         = 2            # Pre-buffer batches in background
    use_channels_last       = True         # ⚡ NHWC Tensor Cores Acceleration
    use_data_parallel       = False         # ⚡ Enable Multi-GPU DataParallel if >= 2 GPUs
    use_amp                 = True         # ⚡ PyTorch Mixed Precision (FP16 AMP)

    # ── ⚡ EXP 1.2: Anti-Overfitting Learning Schedule ────────────────────
    epochs                  = 11           # ⚡ Optimized 10 epochs schedule (Cosine annealing lands on peak)
    pct_start               = 0.20         # ⚡ 20% Warmup then steep cosine decay
    lr_backbone             = 2.8e-5       # ⚡ Anti-Overfitting: moderated learning rate for ConvNeXt backbone
    lr_head                 = 1.5e-3       # Boosted for Cross-Plane & Finding-Specific Head
    lr_prior                = 5.0e-3       # 🧠 Dedicated Fast LR for Anatomical Prior Drift
    unfreeze_last           = 2            # ⚡ Freeze early stages 0-1, unfreeze top 2 ConvNeXt stages (stages 2 & 3)
    weight_decay            = 0.05         # ⚡ Boosted to 0.05 for enhanced L2 anti-overfitting regularization
    max_grad_norm           = 1.0          # Gradient clipping ceiling
    dropout                 = 0.35         # ⚡ Boosted to 0.35 in Finding-Specific Head
    top_k_swa               = 3            # Average weights of top-3 validation epochs
    use_swa                 = True         # Stochastic Weight Averaging
    ema_decay               = 0.997        # EMA decay factor
    use_ema                 = True         # Exponential Moving Average of weights
    early_stopping_patience = 2            # Stop after 2 epochs without val_auc improvement
    early_stopping_min_delta = 0.0003      # Minimum AUC improvement delta to reset early stopping
    min_epoch_early_stop    = 6            # Don't early stop before epoch 6

    # ── 🌟 EXP 3.2: Attention-Masked Whole-Plane Dropout ──────────────────
    use_plane_dropout       = True         # 🌟 Mask an entire view with -1e4 attention logits (Forces multi-plane synthesis)
    plane_dropout_prob      = 0.20         # ⚡ Boosted to 20% to force robust multi-plane representations

    # ── 🌟 Auxiliary Multi-Plane Contrast Classification Task ────────────
    use_contrast_aux        = True         # 🌟 Enable Auxiliary Contrast Prediction Task (Implicit Regularization)
    aux_contrast_weight     = 0.10         # Fixed weight for auxiliary cross-entropy loss (0.05 - 0.10, baseline match)
    n_contrast_types        = 4            # 4 contrast classes: 0=OTHER, 1=FS, 2=T1, 3=NOFS_FLUID
    use_contrast_embedding  = False        # Additive feature embedding (False when aux head is used)
    use_dynamic_contrast_gating = False        # 🌟 Contrast-Aware Dynamic Inter-Plane Gating (Disabled)
    dynamic_plane_gamma         = 1.0          # Multiplier for contrast-driven plane gating logits

    # ── 🧬 13-Dim Tabular Patient & Scanner Metadata Injection ─────────
    use_metadata_injection  = False        # 🌟 Disabled Tabular Metadata (Pure Image Learning to match Colab Fold 0)
    use_meta_film           = False        # 🛡️ Disabled: Pruned 768-dim visual FiLM injection (prevents visual feature pollution)
    meta_dim                = 13           # Sex (2) + PixelSpacing (2) + Padded (1) + Slices (3) + Laterality (1) + MFS_3T (1) + Suboptimal (3)
    meta_dropout_prob       = 0.15         # 🛡️ Modality Dropout: 15% chance to drop metadata (forces heavy visual reliance)
    meta_mlp_dropout        = 0.20         # 🛡️ MLP dropout anti-memorization
    meta_logit_bias_weight  = 0.06         # Logit bias weight
    meta_weight_decay       = 0.05         # ⚡ Synchronized to 0.05 L2 penalty

    # ── 🌟 EXP 5: Stage 2 Soft Label Refinement (70% LLM + 30% OOF) ─────
    use_label_refinement    = False        # ⚡ DISABLED: Direct benchmark of upgraded dataset without refinement
    refinement_alpha        = 0.30         # 30% Vision OOF + 70% Text LLM (Sweet Spot)
    oof_predictions_path = "./checkpoints/oof_predictions_kaggle_5folds.csv"
    use_oof_denoising       = False        # Denoising disabled in favor of soft refinement
    oof_csv_path            = "./oof_predictions_v1.csv"
    denoise_discrepancy_th  = 0.70
    pu_label_clamp_min      = 0.03         # Asymmetric PU clamping for unmentioned lesions
    pu_label_clamp_max      = 0.97         # Prevents infinite logit explosion

    # ── 🌟 Dual-Level Attention Architecture ──────────────────────────────
    head_type               = "cross_plane_finding_specific"
    use_cross_plane         = False        # 🌟 Disabled: preserve view-specific locality for Hierarchical Head
    use_intra_plane_depth   = False        # 🛡️ Disabled: Pruned dead-weight & eliminates 10% dropout noise
    depth_interact_dropout  = 0.10         # Depth residual dropout
    cross_plane_heads       = 4            # 4 Multi-Head Attention Channels
    cross_plane_dropout     = 0.15         # Cross-Plane Dropout
    use_cosine_classifier   = True         # 🌟 Hyperspherical Cosine Metric Head with Shared Scalar Scale
    use_cutout              = False        # 🚫 Cutout disabled to preserve focal lesions

    # ── 🧬 Feature Pooling & Model Hyperparameters ─────────────────────────
    feature_pool            = "convnext_768" # ⚡ ConvNeXt-Tiny stage4 channels = 768 dims
    drop_path_rate          = 0.08         # 🌟 Stochastic Depth: Regularize Unfrozen Stage 2 & 3
    use_multi_prototype     = False        # 🌟 Single-Prototype Cosine Head (K=1 canonical prototype per pathology)
    num_prototypes          = 1            # K=1 canonical prototype prevents overfitting on subtle noise
    use_dual_pooling        = True         # 🌟 Dual-Pooling: Attentive Context + Sharp Focal Peak Slice
    # ── 🌟 EXP: Bilateral Finding Relation Head (DISABLED) ────────────────
    use_finding_relation        = False        # 🌟 Bilateral Finding Relation Head disabled
    use_bilateral_relation      = False        # Alias
    relation_num_heads          = 4            # Multi-head attention across 12 findings
    relation_dropout            = 0.10         # Dropout on relation attention
    relation_gamma_init         = 0.05         # 🌟 Active warm-up init (allows finding attention gradients to flow immediately)
    lr_gates                    = 2.0e-3       # Fast LR for scalar residual gates (Zero Weight Decay)
    lr_relation                 = 1.5e-3       # Relation head learning rate
    use_plane_decoupled_pooling = True     # 🌟 Plane-Decoupled Hierarchical Dual-Pooling (Intra-Plane + Inter-Plane Gating)
    broad_temperature       = 2.0          # 🌟 Dual-Scale Context: soft smoothed temperature capturing broad joint context
    focal_temperature       = 2.0          # Alias for backwards compatibility

    # ── 🌪️ Continuous Depth Jittering & Augmentation ──────────────────────
    use_depth_jitter        = True         # ⚡ Continuous 3D Depth Jittering
    # ── 🩻 2.5D Slice Drop Augmentation ──────────────────────────────────
    use_slice_drop          = True         # 🌟 2.5D Channel Slice Drop (Regularizes adjacent slice dependency)
    slice_drop_prob         = 0.05         # Softened to 5% to preserve 3.5mm physical depth span
    # ── 🩻 MRI Physics: Non-Linear Gamma Contrast Simulation ─────────────
    use_gamma_aug           = True         # 🌟 Random Gamma Power-Law Augmentation (MRI TR/TE contrast simulation)
    gamma_prob              = 0.40         # Apply to 40% of train samples
    gamma_lo                = 0.85         # Brightens soft-tissue & fluid signals
    gamma_hi                = 1.20         # Darkens background noise, highlights bone edges

    # ── 🌟 EXP 2B: GPU Spatial Affine Jittering ─────────────────────────
    use_affine_aug          = True         # 🌟 Upgrade simple rotation to full GPU Affine (Rotate + Shift + Scale)
    affine_prob             = 0.60         # 60% chance to apply spatial affine transform
    affine_max_rotate       = 6.0          # +/- 6.0 degrees rotation
    affine_max_translate    = 0.03         # +/- 3% translation (+/- 8px on 256x256)
    affine_scale_range      = (0.95, 1.05) # 0.95 - 1.05 natural scaling
    depth_jitter_prob       = 0.15         # Softened to 15% for widened coronal band
    depth_jitter_max_shift  = 0.10         # Gentle shift to prevent focal lesion blurring
    mixup_alpha             = 0.30         # 2.5D Multi-Plane Mixup alpha
    mixup_prob              = 0.60         # Apply Mixup to 60% of train batches

    # ── 🌟 Two-Stage Cross-Modal Denoising & Self-Correction Pipeline ──
    run_two_stage_pipeline  = True         # When True, executes Stage 1 -> Self-Correction -> Stage 2 end-to-end
    stage1_confident_only   = True         # Stage 1: Per-Finding Loss Masking on ambiguous text labels (w <= 0.3)
    min_train_confidence    = 0.30         # Cutoff threshold for Stage 1 masking
    self_correct_threshold  = 0.40         # Tau_vis threshold to rescue missing text labels (2*|p_vis - 0.5| >= 0.40)
    self_correct_uncertain_w = 0.05        # Suppress ambiguous labels when both text & vision are uncertain
    refined_labels_save_path = "./checkpoints/llm_labels_v5_refined.csv"
    stage1_epochs           = 11           # Stage 1: Pure visual pretraining
    stage2_epochs           = 11           # Stage 2: Champion retraining on refined ground truth

    # ── ⚖️ Class-Calibrated Soft PU Loss & Continuous Dynamics ────────────
    use_asl                 = True         # Protected Asymmetric Loss Base
    asl_gamma_pos           = 0.0          # Standard CE on positive findings
    asl_gamma_neg           = 2.0          # Suppresses easy negative gradients
    asl_clip                = 0.05         # Asymmetric margin clipping
    rank_loss_w             = 0.15         # Soft-Margin Rank Loss Weight
    gold_weight             = 3.0          # Sample weight multiplier for Gold Standard cases

    # ── Architecture & Model Setup ─────────────────────────────────────────
    backbone                = MODEL_PATH
    n_folds                 = 5
    train_folds             = [0, 1, 2, 3, 4] # 🚀 FULL 5-FOLD KAGGLE GPU TRAINING
    device                  = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # ── 🌟 Clinical & Gaussian-Smoothed Inductive Priors ──────────────────
    use_plane_embedding     = False        # 🛡️ Disabled View-Aware Plane Embeddings (prevents additive Gaussian noise)
    use_laterality_flip     = True         # Base laterality normalization (Right "R" -> Left "L")
    use_slot_prior          = True         # 🌟 Warm-Start Residual Continuous Gaussian Prior (ON)
    slot_prior_strength     = 0.55         # Peak focal boost (+0.55 logit)
    slot_prior_sigma        = 0.90         # 🌟 1D Gaussian kernel standard deviation for smoothing

    # ── Direct Local SSD File Paths ────────────────────────────────────────
    train_csv            = TRAIN_CSV
    llm_dataset_slug     = "stevenleehans/rsna-knee-llm-report-labels"
    llm_labels_csv       = LLM_CSV
    preprocessed_npy     = NPY_PATH
    preprocessed_meta    = META_CSV
    oof_predictions_path = "./checkpoints/oof_predictions_kaggle_5folds.csv"

    # ── 3c. Spatial Pooling & Multi-Stage Feature Fusion ───────────────────
    use_gem_pooling         = True         # Generalized Mean Pooling on ConvNeXt 2D feature maps
    use_fpn_fusion          = True         # 🌟 SOTA 2D Top-Down Channel-Gated FPN Spatial Fusion
    fpn_gamma_init          = 0.05         # 🌟 Active warm-up init (matches baseline, enables gradient flow)
    lr_fpn                  = 1.5e-3       # Fast LR for FPN projection & smoothing layers
    gem_p                   = 3.0          # Initial power: 1.0=AvgPool, 3.0=GeM, inf=MaxPool
    gem_p_trainable         = True         # Learn optimal spatial pooling power end-to-end
    lr_gem                  = 1.0e-3       # Specialized LR for scalar p (Zero Weight Decay)
    # ── Safe Architectural Aliases ────────────────────────────────────────
    use_hierarchical_pooling    = True         # 🌟 Decouples Intra-Plane Slices from Inter-Plane Weighting
    use_plane_decoupled_pooling = True         # Alias for checkpoint compatibility
    use_stage3_fusion           = False        # 🛡️ Disabled 1D Stage3 Vector Fusion
    stage3_fusion_init          = 0.05         # Alias for stage3_gamma_init
    use_trainable_gem           = True         # Alias for use_gem_pooling
    use_multi_stage_fusion  = False        # 🛡️ Disabled 1D Multi-Stage Vector Fusion (Eliminates stage3_gamma & pool_stage3)
    stage3_feat_dim         = 384          # ConvNeXt-Tiny Stage 3 channels
    stage3_gamma_init           = 0.05         # 🌟 Initialized at 0.05 (gentle active injection from Epoch 1, preserves stability)
    lr_stage3_proj          = 1.5e-3       # Fast LR for Stage 3 projection layer (matches head LR)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
        if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
            torch.backends.cuda.matmul.allow_tf32 = True
        if hasattr(torch.backends.cudnn, "allow_tf32"):
            torch.backends.cudnn.allow_tf32 = True

seed_everything(CFG.seed)
print(f"⚡ Device: {CFG.device} | Head: {CFG.head_type.upper()} | Resolution: {CFG.img_size}x{CFG.img_size} | TrainBatch: {CFG.batch_size} ValBatch: {getattr(CFG, 'val_batch_size', 32)}")
print(f"🧬 Feature Pooling: {CFG.feature_pool} (768 dims) | Cross-Plane Fusion: {CFG.use_cross_plane} ({CFG.cross_plane_heads} heads) | Folds: {CFG.train_folds}")
print(f"🎯 Classifier Head: Single-Prototype Cosine + Dual-Pooling (Broad τ={getattr(CFG, 'broad_temperature', 2.0)}) | Finding Relation Head: {getattr(CFG, 'use_finding_relation', False)} (Heads={getattr(CFG, 'relation_num_heads', 4)}, Init_γ={getattr(CFG, 'relation_gamma_init', 0.0)})")
print(f"🛡️ EXP 3.2: Attention-Masked Whole-Plane Dropout = {CFG.use_plane_dropout} (p={CFG.plane_dropout_prob}) | GPU Affine = {CFG.use_affine_aug} (p={CFG.affine_prob})")
print(f"🧬 Metadata Injection: Dim={CFG.meta_dim}, ModalityDrop={CFG.meta_dropout_prob}, MLPDrop={CFG.meta_mlp_dropout}, BiasWeight={CFG.meta_logit_bias_weight} | FiLM={getattr(CFG, 'use_meta_film', False)}")
print(f"⚖️ Loss: SOTA ASL (gamma_neg={CFG.asl_gamma_neg}) + PU Loss + Soft-Margin Rank Loss (w={CFG.rank_loss_w})")
print(f"🧠 Prior LR: {CFG.lr_prior} (Zero Weight Decay) | DropPath: {getattr(CFG, 'drop_path_rate', 0.08)}")
print(f"⚡ Schedule: Epochs={CFG.epochs}, Pct_Start={CFG.pct_start}, WeightDecay={CFG.weight_decay} | Early Stopping = {CFG.early_stopping_patience}")
print(f"🏆 SWA Top-K: {CFG.top_k_swa} epochs averaging")
print(f"🚀 Two-Stage Pipeline: {CFG.run_two_stage_pipeline} | Stage 1 Masking: {CFG.stage1_confident_only} (w <= {CFG.min_train_confidence}) | Self-Correct Tau: {CFG.self_correct_threshold}")
print(f"🌊 Pooling: GeM Pooling (initial p={CFG.gem_p}, trainable={CFG.gem_p_trainable}) | FPN 2D Top-Down Fusion: {getattr(CFG, 'use_fpn_fusion', False)} (Init_γ={getattr(CFG, 'fpn_gamma_init', 0.0)}) | Stage3 dim={getattr(CFG, 'stage3_feat_dim', 384)}")


⚡ Device: cuda:0 | Head: CROSS_PLANE_FINDING_SPECIFIC | Resolution: 224x224 | TrainBatch: 16 ValBatch: 64
🧬 Feature Pooling: convnext_768 (768 dims) | Cross-Plane Fusion: False (4 heads) | Folds: [0, 1, 2, 3, 4]
🎯 Classifier Head: Single-Prototype Cosine + Dual-Pooling (Broad τ=2.0) | Finding Relation Head: False (Heads=4, Init_γ=0.05)
🛡️ EXP 3.2: Attention-Masked Whole-Plane Dropout = True (p=0.2) | GPU Affine = True (p=0.6)
🧬 Metadata Injection: Dim=13, ModalityDrop=0.15, MLPDrop=0.2, BiasWeight=0.06 | FiLM=False
⚖️ Loss: SOTA ASL (gamma_neg=2.0) + PU Loss + Soft-Margin Rank Loss (w=0.15)
🧠 Prior LR: 0.005 (Zero Weight Decay) | DropPath: 0.08
⚡ Schedule: Epochs=11, Pct_Start=0.2, WeightDecay=0.05 | Early Stopping = 2
🏆 SWA Top-K: 3 epochs averaging
🚀 Two-Stage Pipeline: True | Stage 1 Masking: True (w <= 0.3) | Self-Correct Tau: 0.4
🌊 Pooling: GeM Pooling (initial p=3.0, trainable=True) | FPN 2D Top-Down Fusion: True (Init_γ=0.05) | Stage3 dim=384


---
## 2. Load Soft Labels (`v4_blend`), OOF Denoising & Stratified 5-Fold Splitting

- **🛡️ OOF Label Denoising**:
  - Đối với các mẫu non-gold trên tập Train: tính $\Delta = |y_{\text{LLM}} - \hat{y}_{\text{OOF}}|$.
  - Nếu $\Delta > 0.75$: hạ `confidence_weight` về $0.10$ và làm mềm nhãn về $0.50$ (Uncertain).
  - **58 ca Gold Standard**: Giữ nguyên nhãn chuẩn bác sĩ và trọng số $3.0\times$.
  - **Tập Val**: Giữ nguyên 100% nhãn gốc để đo lường hoàn toàn khách quan.


In [4]:
from sklearn.model_selection import StratifiedKFold, KFold

TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    "Synovitis", "Baker's", 'Contusion', 'Fracture'
]
CONF_COLS = [c + '__conf' for c in TARGET_COLS]

# ── 1. Load LLM Report Labels (v4_blend: stevenleehans/rsna-knee-llm-report-labels) ────────
if not os.path.exists(CFG.llm_labels_csv):
    print(f"⚠️ {CFG.llm_labels_csv} not found, auto-downloading directly from Kaggle (426 KB)...")
    out_dir = os.path.dirname(CFG.llm_labels_csv) if os.path.dirname(CFG.llm_labels_csv) else "."
    os.makedirs(out_dir, exist_ok=True)
    os.system("kaggle datasets download -d stevenleehans/rsna-knee-llm-report-labels -p /content/rsna-knee-llm-report-labels --unzip")
    for cand in [
        "/content/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv",
        "./llm_labels_v4_blend.csv"
    ]:
        if os.path.exists(cand):
            CFG.llm_labels_csv = cand
            break
    if not os.path.exists(CFG.llm_labels_csv):
        try:
            import kagglehub, shutil
            k_path = kagglehub.dataset_download("stevenleehans/rsna-knee-llm-report-labels")
            cand = os.path.join(k_path, "llm_labels_v4_blend.csv")
            if os.path.exists(cand):
                CFG.llm_labels_csv = cand
        except Exception as e:
            print(f"kagglehub error: {e}")

if not os.path.exists(CFG.llm_labels_csv) and os.path.exists(CFG.preprocessed_meta):
    print(f"ℹ️ Using preprocessed metadata as labels fallback: {CFG.preprocessed_meta}")
    CFG.llm_labels_csv = CFG.preprocessed_meta

df = pd.read_csv(CFG.llm_labels_csv)
print(f'✅ Loaded LLM Report Labels (v4_blend): {CFG.llm_labels_csv} ({len(df):,} studies)')

# Save pristine raw labels for unbiased validation evaluation
for c in TARGET_COLS:
    df[f'raw_{c}'] = df[c].values.copy()

# ── 2. Identify 58 Gold Standard Cases Directly from Official train.csv ────────
df['is_gold'] = False
if os.path.exists(CFG.train_csv):
    raw_train_df = pd.read_csv(CFG.train_csv)
    if all(c in raw_train_df.columns for c in TARGET_COLS):
        gold_mask = raw_train_df[TARGET_COLS].notna().all(axis=1)
        gold_df = raw_train_df[gold_mask].copy()
        gold_dict = gold_df.set_index('StudyInstanceUID')[TARGET_COLS].to_dict('index')
        for idx, row in df.iterrows():
            uid = row['StudyInstanceUID']
            if uid in gold_dict:
                df.loc[idx, 'is_gold'] = True
                for c in TARGET_COLS:
                    df.loc[idx, c] = gold_dict[uid][c]
                    df.loc[idx, f'raw_{c}'] = gold_dict[uid][c]
        print(f'✅ Successfully identified and calibrated {df["is_gold"].sum()} Gold Standard studies from {CFG.train_csv}')
else:
    print('ℹ️ train.csv not found locally; all samples treated with soft PU weighting.')

# Continuous confidence weights: 2 * |p - 0.5|
for c in TARGET_COLS:
    df[c + '__conf'] = (2.0 * np.abs(df[c].values - 0.5)).clip(0.0, 1.0)
    # Gold cases have 1.0 confidence
    df.loc[df['is_gold'] == True, c + '__conf'] = 1.0

# ── 3. Native Multilabel Stratified 5-Fold Splitting ─────────────────────────
print('🚀 Generating Native Stratified 5-Fold Split balancing Gold and Pathology prevalence...')
df['fold'] = -1

# Step A: Distribute Gold Standard cases evenly across 5 folds (11-12 per fold)
gold_idxs = df[df['is_gold'] == True].index.values
if len(gold_idxs) > 0:
    gold_pos_sum = (df.loc[gold_idxs, TARGET_COLS] > 0.5).sum(axis=1).values
    skf_gold = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    for f_i, (_, val_pos) in enumerate(skf_gold.split(gold_idxs, gold_pos_sum.clip(0, 3))):
        df.loc[gold_idxs[val_pos], 'fold'] = f_i

# Step B: Multilabel Stratified Split on the remaining non-gold studies
nongold_idxs = df[df['is_gold'] == False].index.values
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
    mskf = MultilabelStratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    bin_labels = (df.loc[nongold_idxs, TARGET_COLS] > 0.5).astype(int).values
    for f_i, (_, val_pos) in enumerate(mskf.split(nongold_idxs, bin_labels)):
        df.loc[nongold_idxs[val_pos], 'fold'] = f_i
    print('✅ Applied MultilabelStratifiedKFold successfully.')
except ImportError:
    # Robust fallback: Multi-target stratification key based on combined pathology severity and counts
    pos_matrix = (df.loc[nongold_idxs, TARGET_COLS] > 0.5).astype(int)
    # Primary stratifier: Total pathology count + presence of common lesions (ACL, Meniscus, OA)
    strat_key = pos_matrix.sum(axis=1).clip(0, 4) * 10 + (pos_matrix['ACL'] * 4 + pos_matrix['Medial Meniscus'] * 2 + pos_matrix['Medial OA'])
    skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    for f_i, (_, val_pos) in enumerate(skf.split(nongold_idxs, strat_key)):
        df.loc[nongold_idxs[val_pos], 'fold'] = f_i
    print('✅ Applied StratifiedKFold multi-lesion key successfully.')

print(f'✅ Active Stratified 5-Fold splits (Seed: {CFG.seed})')
print(f'   Total studies per fold: {df["fold"].value_counts().sort_index().to_dict()}')
gold_dist = df[df['is_gold']]['fold'].value_counts().sort_index().to_dict()
print(f'   🌟 Gold Standard cases per fold: {gold_dist}')

# ── 4. Dynamic In-Fold Auto-Calibrated PU & FN Weights (100% Zero Data Leakage) ──
def compute_auto_pu_fn_weights_per_fold(train_df, target_cols, conf_cols, device=CFG.device):
    """
    🧠 Tự động tính toán PU Weights và FN Penalty Weights từ thống kê nội bộ Fold.
    - 100% Data Leakage Free: Chỉ chạy trên tập train_df của fold hiện tại, không nhìn thấy val_df.
    - FN Penalty: Hàm tỷ lệ nghịch tần suất có giảm chấn (Damped Inverse Frequency). Bệnh càng hiếm, FN penalty càng cao.
    - PU Weight: Tính trực tiếp từ độ tin cậy trung bình của tập nhãn âm tính (Negative Pool Confidence).
    """
    auto_fn_weights = []
    auto_pu_weights = []
    
    for c, c_conf in zip(target_cols, conf_cols):
        y = train_df[c].values          # Nhãn mềm [0, 1]
        w = train_df[c_conf].values     # Độ tin cậy [0, 1]
        
        # Tần suất dương tính thực nghiệm của lớp c trong fold này
        p_c = float(np.clip(np.mean(y), 1e-4, 1.0 - 1e-4))
        
        # 🌟 A. AUTO FN PENALTY (Damped Inverse Frequency: fn_w = ((1 - p_c) / p_c) ^ 0.20)
        imbalance_ratio = (1.0 - p_c) / p_c
        fn_w = float(np.clip(np.power(imbalance_ratio, 0.20), 1.00, 1.60))
        auto_fn_weights.append(fn_w)
        
        # 🌟 B. AUTO PU WEIGHT (Negative Pool Confidence)
        neg_mask = (y < 0.5)
        if neg_mask.sum() > 0:
            avg_neg_conf = float(np.mean(w[neg_mask]))
        else:
            avg_neg_conf = 0.50
            
        pu_w = float(np.clip(avg_neg_conf * (1.0 - 0.35 * p_c), 0.15, 0.70))
        auto_pu_weights.append(pu_w)
        
    fn_tensor = torch.tensor(auto_fn_weights, dtype=torch.float32, device=device).view(1, -1)
    pu_tensor = torch.tensor(auto_pu_weights, dtype=torch.float32, device=device).view(1, -1)
    return auto_fn_weights, auto_pu_weights, fn_tensor, pu_tensor

# Safe Initial Fallback (Calculated dynamically, will be recalibrated per fold in training loop)
auto_fn_list, auto_pu_list, FN_WEIGHTS_TENSOR, PU_WEIGHTS_TENSOR = compute_auto_pu_fn_weights_per_fold(df, TARGET_COLS, CONF_COLS)


✅ Loaded LLM Report Labels (v4_blend): /kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv (4,407 studies)
✅ Successfully identified and calibrated 58 Gold Standard studies from /kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv
🚀 Generating Native Stratified 5-Fold Split balancing Gold and Pathology prevalence...
✅ Applied MultilabelStratifiedKFold successfully.
✅ Active Stratified 5-Fold splits (Seed: 42)
   Total studies per fold: {0: 882, 1: 881, 2: 882, 3: 881, 4: 881}
   🌟 Gold Standard cases per fold: {0: 12, 1: 12, 2: 12, 3: 11, 4: 11}


---
## 3. Fast Master Array Loader & Denoised KneeMRIDataset


In [5]:
# ── Load Master Preprocessed Array ───────────────────────────────────────────
mmap_data = np.load(CFG.preprocessed_npy, mmap_mode="r")
print(f"🚀 Loaded Master Array via zero-copy mmap: {CFG.preprocessed_npy} | Shape: {mmap_data.shape}")

# ── Extract Laterality, Contrast Flags & 9-Dim Tabular Metadata ─────────────
meta_df = pd.read_csv(CFG.preprocessed_meta)
uid_to_mmap_idx = {uid: i for i, uid in enumerate(meta_df["StudyInstanceUID"])}
study_lat_map = meta_df.groupby("StudyInstanceUID")["Laterality"].first().to_dict()

# Extract 4-Way Contrast Types (FS, T1, NOFS_FLUID, OTHER)
CONTRAST_MAP = {"OTHER": 0, "FS": 1, "T1": 2, "NOFS_FLUID": 3, "UNK": 0}
study_contrast_map = {}
sub_meta = meta_df[["StudyInstanceUID", "sag_contrast", "cor_contrast", "axi_contrast"]].drop_duplicates("StudyInstanceUID")
for _, r in sub_meta.iterrows():
    s_c = CONTRAST_MAP.get(str(r["sag_contrast"]).upper(), 0)
    c_c = CONTRAST_MAP.get(str(r["cor_contrast"]).upper(), 0)
    a_c = CONTRAST_MAP.get(str(r["axi_contrast"]).upper(), 0)
    s_p = getattr(CFG, "num_slices_per_plane", 7)
    study_contrast_map[r["StudyInstanceUID"]] = [s_c]*s_p + [c_c]*s_p + [a_c]*s_p

def safe_float(val, default):
    try:
        if val is None or pd.isna(val):
            return default
        v = float(val)
        return default if math.isnan(v) else v
    except (ValueError, TypeError):
        return default

# Extract 13-Dim Tabular Metadata (with 3T Magnetic Field Strength & Suboptimal Flags)
study_meta_map = {}
for _, r in meta_df.drop_duplicates("StudyInstanceUID").iterrows():
    uid = r["StudyInstanceUID"]
    sex_m = 1.0 if str(r.get("PatientSex", "")).upper() == "M" else 0.0
    sex_f = 1.0 if str(r.get("PatientSex", "")).upper() == "F" else 0.0
    ps_r  = float(np.clip((safe_float(r.get("PixelSpacing_Row"), 0.45) - 0.45) / 0.15, -3.0, 3.0))
    ps_c  = float(np.clip((safe_float(r.get("PixelSpacing_Col"), 0.45) - 0.45) / 0.15, -3.0, 3.0))
    cp    = 1.0 if int(safe_float(r.get("Crop_Padded"), 0)) == 1 else 0.0
    sag_s = float(np.clip((safe_float(r.get("sag_slices"), 25.0) - 25.0) / 10.0, -2.0, 2.0))
    cor_s = float(np.clip((safe_float(r.get("cor_slices"), 25.0) - 25.0) / 10.0, -2.0, 2.0))
    axi_s = float(np.clip((safe_float(r.get("axi_slices"), 25.0) - 25.0) / 10.0, -2.0, 2.0))
    is_r  = 1.0 if str(r.get("Laterality", "L")).upper() == "R" else 0.0
    # 🌟 Scanner 3.0T vs 1.5T Magnetic Field Strength & Suboptimal Plane Flags
    mfs   = safe_float(r.get("MagneticFieldStrength"), 1.5)
    is_3t = 1.0 if mfs >= 2.5 else 0.0
    s_sub = 1.0 if int(safe_float(r.get("sag_is_suboptimal"), 0)) == 1 else 0.0
    c_sub = 1.0 if int(safe_float(r.get("cor_is_suboptimal"), 0)) == 1 else 0.0
    a_sub = 1.0 if int(safe_float(r.get("axi_is_suboptimal"), 0)) == 1 else 0.0
    study_meta_map[uid] = np.array([sex_m, sex_f, ps_r, ps_c, cp, sag_s, cor_s, axi_s, is_r, is_3t, s_sub, c_sub, a_sub], dtype=np.float32)

print(f"✅ Extracted {CFG.meta_dim}-Dim Metadata & Contrast for {len(study_meta_map):,} studies directly from {CFG.preprocessed_meta}")

# 🛡️ Align and Guard DataFrame against Index Mismatches
if uid_to_mmap_idx:
    initial_len = len(df)
    df = df[df["StudyInstanceUID"].isin(uid_to_mmap_idx)].copy()
    df["study_idx"] = df["StudyInstanceUID"].map(uid_to_mmap_idx).astype(int)
    print(f"✅ Synchronized Label DataFrame: {len(df):,}/{initial_len:,} studies matched to Master Array")
else:
    max_safe_len = min(len(df), len(mmap_data))
    df = df.iloc[:max_safe_len].copy()
    df["study_idx"] = np.arange(len(df))

df = df.reset_index(drop=True)

IMAGENET_MEAN = torch.tensor([0.450, 0.450, 0.450]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.225, 0.225, 0.225]).view(1, 3, 1, 1)

# ── ⚡ 100% Zero-CPU-Overhead Vectorized KneeMRIDataset (uint8 Direct Transfer) ─
class KneeMRIDataset(Dataset):
    def __init__(self, df, mmap_data, lat_map=None, contrast_map=None, meta_map=None, is_train=True):
        self.df           = df.reset_index(drop=True)
        self.mmap_data    = mmap_data
        self.lat_map      = lat_map if lat_map is not None else {}
        self.contrast_map = contrast_map if contrast_map is not None else {}
        self.meta_map     = meta_map if meta_map is not None else {}
        self.is_train     = is_train

        # ⚡ 1. Extract Labels
        if not is_train:
            raw_cols = [f"raw_{c}" for c in TARGET_COLS]
            if all(c in self.df.columns for c in raw_cols):
                labels = self.df[raw_cols].values.astype(np.float32).copy()
            else:
                labels = self.df[TARGET_COLS].values.astype(np.float32).copy()
        else:
            labels = self.df[TARGET_COLS].values.astype(np.float32).copy()
            is_gold_arr = self.df["is_gold"].values.astype(bool) if "is_gold" in self.df.columns else np.zeros(len(self.df), dtype=bool)
            non_gold_mask = ~is_gold_arr
            labels[non_gold_mask] = np.clip(labels[non_gold_mask], CFG.pu_label_clamp_min, CFG.pu_label_clamp_max)

        conf_cols = [c + "__conf" for c in TARGET_COLS]
        if all(c in self.df.columns for c in conf_cols):
            weights = self.df[conf_cols].values.astype(np.float32).copy()
        else:
            weights = np.clip(2.0 * np.abs(labels - 0.5), 0.0, 1.0).astype(np.float32)

        # ⚡ 3. Apply Gold Standard Sample Weight Multiplier (3.0x)
        if "is_gold" in self.df.columns:
            gold_bool = self.df["is_gold"].values.astype(bool)
            weights[gold_bool] = float(CFG.gold_weight)

        # 🎯 STAGE 1 PER-FINDING LOSS MASKING:
        # Completely zero out loss gradients on ambiguous non-gold text labels (w <= min_train_confidence)
        # STRICT ZERO LEAKAGE: All 58 Gold Standards are strictly protected (w = 3.0)!
        if self.is_train and getattr(CFG, "stage1_confident_only", False):
            min_conf = getattr(CFG, "min_train_confidence", 0.30)
            is_gold_vec = self.df["is_gold"].values.astype(bool) if "is_gold" in self.df.columns else np.zeros(len(self.df), dtype=bool)
            noisy_mask = (~is_gold_vec[:, None]) & (weights <= min_conf)
            weights[noisy_mask] = 0.0

        self.precomputed_labels  = labels
        self.precomputed_weights = weights
        self.study_indices       = self.df["study_idx"].values.astype(int) if "study_idx" in self.df.columns else np.arange(len(self.df))
        self.is_gold_arr         = self.df["is_gold"].values.astype(bool) if "is_gold" in self.df.columns else np.zeros(len(self.df), dtype=bool)
        self.study_uids          = self.df["StudyInstanceUID"].values
        # ⚡ Precompute laterality flags: zero CPU lookups in __getitem__
        self.is_right_arr        = np.array([self.lat_map.get(uid, "L") == "R" for uid in self.study_uids], dtype=bool)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        study_id = self.study_uids[idx]
        study_ix = self.study_indices[idx]
        is_gold  = bool(self.is_gold_arr[idx])
        contrast_flags = self.contrast_map.get(study_id, [0]*CFG.num_slices)
        default_meta = np.zeros(getattr(CFG, "meta_dim", 13), dtype=np.float32)
        default_meta[0:2] = 0.5
        meta_vec = self.meta_map.get(study_id, default_meta)
        is_right = bool(self.is_right_arr[idx])

        # Direct zero-copy uint8 mmap slice (only 2.9 MB per sample, zero CPU tensor transforms)
        sampled = np.array(self.mmap_data[study_ix])
        t_slices = torch.from_numpy(sampled)

        labels  = torch.from_numpy(self.precomputed_labels[idx])
        weights = torch.from_numpy(self.precomputed_weights[idx])
        c_flags = torch.tensor(contrast_flags, dtype=torch.long)
        meta_t  = torch.from_numpy(meta_vec).float()

        # ⚡ Returns raw uint8 tensor + is_right flag directly; GPU performs laterality flip & aug in parallel (< 0.05ms)
        return t_slices, labels, weights, is_gold, c_flags, meta_t, is_right


🚀 Loaded Master Array via zero-copy mmap: /tmp/knee_train_21slices_224.npy | Shape: (4407, 21, 3, 224, 224)
✅ Extracted 13-Dim Metadata & Contrast for 4,407 studies directly from /kaggle/input/datasets/mediumguy/rsna-preprocessing-data/train_metadata_21slices.csv
✅ Synchronized Label DataFrame: 4,407/4,407 studies matched to Master Array


---
## 4. ConvNeXt-Tiny 2.5D with Cross-Plane Fusion & Finding-Specific Attention Head


In [6]:
class WarmStartResidualGaussianSlotPrior(nn.Module):
    """
    🧠 Warm-Start Residual Anatomical Gaussian Slot Prior with Dedicated Geometry Learning.
    - Anchors: Initialized at verified clinical landmarks (mu_0, sigma_0, pi_0).
    - Bounded Residual Shifts: delta_mu, delta_sigma, delta_gate clamped via tanh.
    - Plane Gating: Learns which orthogonal plane is dominant for each pathology.
    """
    def __init__(self, n_classes=12, n_slices_per_plane=7):
        super().__init__()
        self.n_classes = n_classes
        self.n_slices_per_plane = n_slices_per_plane
        self.total_slices = n_slices_per_plane * 3

        # Normalized anchors mu_0, sigma_0, pi_0 for (12 classes, 3 planes: 0=Sagittal, 1=Coronal, 2=Axial)
        # Slices 0..6 normalized as s/6.0 in [0, 1]
        # Normalized anchors mu_0, sigma_0, pi_0 for (12 classes, 3 planes: 0=Sagittal, 1=Coronal, 2=Axial)
        # Slices 0..6 normalized as s/6.0 in [0, 1]
        # Anatomical alignment:
        # Sagittal: Medial (low index ~1.5) <-> Lateral (high index ~4.5) on normalized Left Knee standard.
        # Coronal: Anterior (front index ~1.5) <-> Posterior (back index ~4.5).
        # Axial: Anterior (patellofemoral ~1.5) <-> Posterior (popliteal/Baker ~4.8).
        anchors_mu = [
            [3.0/6.0, 3.0/6.0, 3.0/6.0], # ACL: Sagittal center, Coronal center
            [1.2/6.0, 3.5/6.0, 3.0/6.0], # MCL: Medial side in Sagittal, joint center in Coronal
            [1.5/6.0, 3.8/6.0, 3.0/6.0], # Medial Meniscus: Medial in Sagittal, central/posterior horn in Coronal
            [4.5/6.0, 3.8/6.0, 3.0/6.0], # Lateral Meniscus: Lateral in Sagittal, central/posterior horn in Coronal
            [1.5/6.0, 3.5/6.0, 3.0/6.0], # Medial OA: Medial in Sagittal, joint line in Coronal
            [4.5/6.0, 3.5/6.0, 3.0/6.0], # Lateral OA: Lateral in Sagittal, joint line in Coronal
            [3.0/6.0, 1.5/6.0, 1.5/6.0], # PF OA: Anterior in Coronal & Axial (patellofemoral joint)
            [3.0/6.0, 2.0/6.0, 1.5/6.0], # Effusion: Suprapatellar pouch (Anterior in Coronal & Axial)
            [3.0/6.0, 2.5/6.0, 2.0/6.0], # Synovitis: Anterior/central joint capsule
            [1.8/6.0, 4.8/6.0, 4.8/6.0], # Baker's: Posteromedial gastrocnemius-semimembranosus bursa (Posterior Coronal & Axial)
            [3.0/6.0, 3.0/6.0, 3.0/6.0], # Contusion: Multi-plane joint center
            [3.0/6.0, 3.0/6.0, 3.0/6.0], # Fracture: Multi-plane joint center
        ]

        anchors_sigma = [
            [0.25, 0.30, 0.30], # ACL
            [0.30, 0.25, 0.30], # MCL
            [0.20, 0.25, 0.30], # Medial Meniscus
            [0.20, 0.25, 0.30], # Lateral Meniscus
            [0.22, 0.25, 0.30], # Medial OA
            [0.25, 0.22, 0.30], # Lateral OA
            [0.25, 0.30, 0.25], # PF OA
            [0.28, 0.35, 0.28], # Effusion
            [0.28, 0.32, 0.28], # Synovitis
            [0.20, 0.30, 0.25], # Baker's
            [0.32, 0.30, 0.30], # Contusion
            [0.28, 0.26, 0.28], # Fracture
        ]

        anchors_pi = [
            [0.85, 0.15, 0.40], # ACL: Sagittal primary
            [0.15, 0.85, 0.40], # MCL: Coronal primary
            [0.85, 0.60, 0.25], # Medial Meniscus: Sagittal primary, Coronal secondary
            [0.85, 0.80, 0.25], # Lateral Meniscus: Sagittal primary, Coronal secondary
            [0.75, 0.90, 0.30], # Medial OA: Coronal primary, Sagittal secondary
            [0.80, 0.75, 0.30], # Lateral OA: Coronal & Sagittal
            [0.70, 0.30, 0.85], # PF OA: Axial & Sagittal
            [0.85, 0.40, 0.70], # Effusion: Sagittal & Axial
            [0.55, 0.80, 0.65], # Synovitis: Coronal & Axial
            [0.25, 0.15, 0.90], # Baker's: Axial primary
            [0.70, 0.75, 0.65], # Contusion: Multi-plane
            [0.70, 0.80, 0.65], # Fracture: Multi-plane
        ]

        self.register_buffer('mu_0', torch.tensor(anchors_mu, dtype=torch.float32))
        self.register_buffer('sigma_0', torch.tensor(anchors_sigma, dtype=torch.float32))
        self.register_buffer('pi_0', torch.tensor(anchors_pi, dtype=torch.float32))

        # Learnable residual parameters initialized to 0
        self.delta_mu    = nn.Parameter(torch.zeros(n_classes, 3, dtype=torch.float32))
        self.delta_sigma = nn.Parameter(torch.zeros(n_classes, 3, dtype=torch.float32))
        self.delta_gate  = nn.Parameter(torch.zeros(n_classes, 3, dtype=torch.float32))

        s_grid = torch.linspace(0.0, 1.0, n_slices_per_plane)
        self.register_buffer('s_grid', s_grid)

    def get_geometry(self):
        mu = torch.clamp(self.mu_0 + 0.25 * torch.tanh(self.delta_mu), 0.05, 0.95)
        sigma = torch.clamp(self.sigma_0 * torch.exp(0.5 * torch.tanh(self.delta_sigma)), 0.08, 1.0)
        pi = torch.clamp(self.pi_0 + 0.30 * torch.tanh(self.delta_gate), 0.05, 0.98)
        return mu, sigma, pi

    def forward(self):
        mu, sigma, pi = self.get_geometry()
        s = self.s_grid.view(1, 1, self.n_slices_per_plane)
        m = mu.unsqueeze(-1)
        sig = sigma.unsqueeze(-1)
        p_val = pi.unsqueeze(-1) * torch.exp(-0.5 * ((s - m) / sig) ** 2)
        p_matrix = p_val.permute(1, 2, 0).reshape(self.total_slices, self.n_classes)
        return p_matrix

    def print_learned_anatomical_centers(self):
        mu, sigma, pi = self.get_geometry()
        mu = mu.detach().cpu().numpy()
        sigma = sigma.detach().cpu().numpy()
        pi = pi.detach().cpu().numpy()
        mu_0 = self.mu_0.cpu().numpy()

        scale_s = float(self.n_slices_per_plane - 1)
        print("\n" + "="*96)
        print("🧠 WARM-START RESIDUAL ANATOMICAL CENTERS & ADAPTIVE LESION GEOMETRY")
        print("="*96)
        print(f"{'Target Pathology':<18} | {'Sagittal (Center [Drift] / Spread / Gate)':<41} | {'Coronal (Center [Drift] / Spread / Gate)'}")
        print("-" * 96)

        for i, c_name in enumerate(TARGET_COLS):
            s_c = mu[i, 0] * scale_s
            s_d = (mu[i, 0] - mu_0[i, 0]) * scale_s
            s_sp = sigma[i, 0] * scale_s
            s_g = pi[i, 0] * 100.0
            s_sign = "+" if s_d >= 0 else ""
            s_str = f"L{s_c:.1f} [{s_sign}{s_d:.1f}sl] (±{s_sp:.1f}sl, {s_g:.0f}%)"

            c_c = mu[i, 1] * scale_s
            c_d = (mu[i, 1] - mu_0[i, 1]) * scale_s
            c_sp = sigma[i, 1] * scale_s
            c_g = pi[i, 1] * 100.0
            c_sign = "+" if c_d >= 0 else ""
            c_str = f"L{c_c:.1f} [{c_sign}{c_d:.1f}sl] (±{c_sp:.1f}sl, {c_g:.0f}%)"

            a_c = mu[i, 2] * scale_s
            a_d = (mu[i, 2] - mu_0[i, 2]) * scale_s
            a_sp = sigma[i, 2] * scale_s
            a_g = pi[i, 2] * 100.0
            a_sign = "+" if a_d >= 0 else ""
            a_str = f"Axial: L{a_c:.1f} [{a_sign}{a_d:.1f}sl] (±{a_sp:.1f}sl, {a_g:.0f}%)"

            print(f"{c_name:<18} | {s_str:<41} | {c_str}")
            print(f"{'':<18} | {a_str}")
            print("-" * 96)


import timm

class GeMPool2d(nn.Module):
    """
    🌊 Generalized Mean (GeM) Pooling (Radenović et al., TPAMI 2018):
    f = (1 / |Omega| * sum(x^p))^(1/p)
    - Adapts smoothly between Average Pooling (p=1) and Max Pooling (p -> inf).
    - Initialized at p=3.0 to focus on localized knee pathology peaks (meniscal tears,
      ACL ruptures, cartilage defects) without diluting the signal with normal joint background.
    """
    def __init__(self, p=3.0, eps=1e-6, p_trainable=True):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p) if p_trainable else torch.tensor(p)
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0, max=10.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=[-2, -1]).pow(1.0 / p)



class ConvNeXtFPN(nn.Module):
    """
    🏗️ Feature Pyramid Network (FPN) Top-Down 2D Spatial Fusion (Baseline Version):
    Fuses high-level semantic context from Stage 4 (7x7) with fine-grained anatomical
    spatial detail from Stage 3 (14x14) in true 2D feature map space BEFORE pooling.

    Architecture:
    1. Lateral Conv 4: 1x1 Conv (768 -> 768) + GroupNorm + GELU
    2. Lateral Conv 3: 1x1 Conv (384 -> 768) + GroupNorm + GELU
    3. Top-Down Pathway: P4_up = Upsample2x(P4) (7x7 -> 14x14)
    4. 2D Spatial Element-wise Sum: P3 = Lateral3(Stage 3) + P4_up
    5. Anti-aliasing Smooth Conv: 3x3 Depthwise Separable Conv on P3
    6. Multi-Scale GeM Pooling: Z4 = GeMPool(P4), Z3 = GeMPool(P3)
    7. Active Residual Blending: Z_fused = Z4 + gamma * Z3 (Scalar gamma init=0.05)
    """
    def __init__(self, in_channels_s3=384, in_channels_s4=768, out_dim=768,
                 gem_p=3.0, gem_p_trainable=True, gamma_init=0.05):
        super().__init__()
        self.out_dim = out_dim

        self.lateral_s4 = nn.Sequential(
            nn.Conv2d(in_channels_s4, out_dim, kernel_size=1, bias=False),
            nn.GroupNorm(num_groups=32, num_channels=out_dim),
            nn.GELU()
        )
        self.lateral_s3 = nn.Sequential(
            nn.Conv2d(in_channels_s3, out_dim, kernel_size=1, bias=False),
            nn.GroupNorm(num_groups=32, num_channels=out_dim),
            nn.GELU()
        )

        self.smooth_s3 = nn.Sequential(
            nn.Conv2d(out_dim, out_dim, kernel_size=3, padding=1, groups=out_dim, bias=False),
            nn.Conv2d(out_dim, out_dim, kernel_size=1, bias=False),
            nn.GroupNorm(num_groups=32, num_channels=out_dim),
            nn.GELU()
        )

        self.pool_p4 = GeMPool2d(p=gem_p, eps=1e-6, p_trainable=gem_p_trainable)
        self.pool_p3 = GeMPool2d(p=gem_p, eps=1e-6, p_trainable=gem_p_trainable)
        self.gamma = nn.Parameter(torch.full((1,), float(gamma_init)))

    def forward(self, stage3_map, stage4_map):
        p4 = self.lateral_s4(stage4_map)  # (N, 768, 7, 7)
        c3 = self.lateral_s3(stage3_map)  # (N, 768, 14, 14)
        p4_up = F.interpolate(p4, size=c3.shape[-2:], mode='bilinear', align_corners=False)  # (N, 768, 14, 14)
        p3 = self.smooth_s3(c3 + p4_up)  # (N, 768, 14, 14)
        z4 = self.pool_p4(p4)  # (N, 768)
        z3 = self.pool_p3(p3)  # (N, 768)
        return z4 + self.gamma * z3


def load_convnext_backbone(model_name=CFG.backbone, drop_path_rate=getattr(CFG, "drop_path_rate", 0.08)):
    print(f"✅ Loading ConvNeXt-Tiny backbone from: {model_name} (DropPath: {drop_path_rate})")
    pretrained_path = None
    if os.path.exists(model_name):
        candidates = [os.path.join(model_name, f) for f in os.listdir(model_name) if f.endswith((".pth", ".pt", ".bin", ".safetensors"))]
        if candidates:
            pretrained_path = candidates[0]
            print(f"   📦 Found local model weights: {pretrained_path}")

    try:
        if pretrained_path:
            model = timm.create_model('convnext_tiny.in12k_ft_in1k', pretrained=False, num_classes=0, drop_path_rate=drop_path_rate)
            if pretrained_path.endswith(".safetensors"):
                from safetensors.torch import load_file
                state_dict = load_file(pretrained_path)
            else:
                state_dict = torch.load(pretrained_path, map_location="cpu")
            if "model" in state_dict: state_dict = state_dict["model"]
            if "state_dict" in state_dict: state_dict = state_dict["state_dict"]
            state_dict = {k: v for k, v in state_dict.items() if not k.startswith("head.")}
            model.load_state_dict(state_dict, strict=False)
            print("   ✅ Loaded state_dict into ConvNeXt-Tiny successfully.")
        else:
            model = timm.create_model('convnext_tiny.in12k_ft_in1k', pretrained=True, num_classes=0, drop_path_rate=drop_path_rate)
            print("   ✅ Loaded pretrained timm/convnext_tiny.in12k_ft_in1k from timm hub.")
    except Exception as e:
        print(f"⚠️ Local load note ({e}). Falling back to timm pretrained...")
        model = timm.create_model('convnext_tiny.in12k_ft_in1k', pretrained=True, num_classes=0, drop_path_rate=drop_path_rate)

    class ConvNeXtWrapper(nn.Module):
        def __init__(self, m, use_gem=getattr(CFG, 'use_gem_pooling', True),
                     gem_p=getattr(CFG, 'gem_p', 3.0),
                     gem_p_trainable=getattr(CFG, 'gem_p_trainable', True),
                     use_fpn_fusion=getattr(CFG, 'use_fpn_fusion', True),
                     use_multi_stage_fusion=getattr(CFG, 'use_multi_stage_fusion', True),
                     stage3_feat_dim=getattr(CFG, 'stage3_feat_dim', 384),
                     fpn_gamma_init=getattr(CFG, 'fpn_gamma_init', 0.05),
                     stage3_gamma_init=getattr(CFG, 'stage3_gamma_init', 0.0)):
            super().__init__()
            self.model = m
            self.embed_dim = 768
            self.use_gem = use_gem
            self.use_fpn_fusion = use_fpn_fusion
            self.use_multi_stage_fusion = use_multi_stage_fusion

            if self.use_gem:
                self.pool = GeMPool2d(p=gem_p, eps=1e-6, p_trainable=gem_p_trainable)
            else:
                self.pool = None

            # 🌟 2D Top-Down Feature Pyramid Network (FPN) Module
            if self.use_fpn_fusion:
                self.fpn = ConvNeXtFPN(
                    in_channels_s3=stage3_feat_dim, in_channels_s4=768, out_dim=768,
                    gem_p=gem_p, gem_p_trainable=gem_p_trainable, gamma_init=fpn_gamma_init
                )
            else:
                self.fpn = None
                if self.use_multi_stage_fusion:
                    self.pool_stage3 = GeMPool2d(p=gem_p, eps=1e-6, p_trainable=gem_p_trainable) if self.use_gem else None
                    self.stage3_proj = nn.Sequential(
                        nn.LayerNorm(stage3_feat_dim),
                        nn.Linear(stage3_feat_dim, 768),
                        nn.GELU()
                    )
                    self.stage3_gamma = nn.Parameter(torch.full((1,), float(stage3_gamma_init)))
                else:
                    self.pool_stage3 = None
                    self.stage3_proj = None
                    self.stage3_gamma = None

        def forward(self, x):
            if (not self.use_fpn_fusion and not self.use_multi_stage_fusion) or not hasattr(self.model, 'stages') or len(self.model.stages) < 4:
                feat_map = self.model.forward_features(x)
                if self.use_gem and self.pool is not None:
                    return self.pool(feat_map)
                else:
                    return feat_map.mean(dim=[-2, -1])

            # Sequential forward through early stages
            h = self.model.stem(x)
            h = self.model.stages[0](h)
            h = self.model.stages[1](h)
            stage3_map = self.model.stages[2](h)  # (N, 384, 14, 14)
            stage4_map = self.model.stages[3](stage3_map)  # (N, 768, 7, 7)
            if hasattr(self.model, 'norm_pre') and self.model.norm_pre is not None:
                stage4_map = self.model.norm_pre(stage4_map)

            # 🌟 2D Top-Down Spatial Fusion Pathway
            if self.use_fpn_fusion and self.fpn is not None:
                return self.fpn(stage3_map, stage4_map)

            # 1D Vector Fallback Pathway
            feat_s4 = self.pool(stage4_map) if (self.use_gem and self.pool is not None) else stage4_map.mean(dim=[-2, -1])
            feat_s3 = self.pool_stage3(stage3_map) if (self.use_gem and hasattr(self, 'pool_stage3') and self.pool_stage3 is not None) else stage3_map.mean(dim=[-2, -1])
            feat_s3_proj = self.stage3_proj(feat_s3)
            return feat_s4 + self.stage3_gamma * feat_s3_proj

    return ConvNeXtWrapper(model)

class CrossPlaneFusionBlock(nn.Module):
    def __init__(self, dim=768, num_heads=CFG.cross_plane_heads, dropout=CFG.cross_plane_dropout):
        super().__init__()
        self.pre_norm = nn.LayerNorm(dim)
        self.cross_attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim)
        )
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, feats):
        h = self.pre_norm(feats)
        attn_out, _ = self.cross_attn(h, h, h)
        x = feats + self.gamma * attn_out
        x = self.norm2(x + self.ffn(self.norm1(x)))
        return x



class IntraPlaneVolumetricDepthBlock(nn.Module):
    """
     SOTA 3D Intra-Plane Volumetric Depth Interaction Block:
    Models true continuous through-plane anatomical connectivity (ACL path, meniscus integrity,
    joint space uniformity) along the slice depth dimension within each isolated imaging plane.
    
    Properties:
    1. Zero Cross-Plane Leakage: Reshapes (B, 3 * S_p, C) -> (B * 3, S_p, C) so planes never cross.
    2. Depthwise 1D Convolution (Kernel=3, Padding=1): Allows slice s to interact strictly with s-1 and s+1.
    3. Channel Gating & LayerNorm: Normalizes feature distribution across depth slices.
    4. Zero-Initialized Residual Gate (gamma = 0.0): Starts at 100% identity to guarantee zero degradation.
    """
    def __init__(self, dim=768, n_planes=3, dropout=0.10):
        super().__init__()
        self.n_planes = n_planes
        self.norm = nn.LayerNorm(dim)
        # Depthwise Conv: kernel_size=3 over slices
        self.depth_conv = nn.Conv1d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False)
        self.pointwise = nn.Sequential(
            nn.Conv1d(dim, dim, kernel_size=1, bias=False),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, feats):
        # feats: (B, S, C) where S = n_planes * S_p
        B, S, C = feats.shape
        S_p = S // self.n_planes
        
        # Reshape strictly within each plane: (B * 3, S_p, C)
        h = feats.view(B * self.n_planes, S_p, C)
        h_norm = self.norm(h).transpose(1, 2) # (B * 3, C, S_p)
        
        # Depthwise 3D through-plane interaction
        out = self.depth_conv(h_norm)
        out = self.pointwise(out) # (B * 3, C, S_p)
        out = out.transpose(1, 2).contiguous().view(B, S, C)
        
        return feats + self.gamma * out

class BilateralFindingRelationHead(nn.Module):
    """
    🦴 Bilateral & Cross-Finding Relational Head:
    1. Bilateral Compartmental Coupling:
       Explicitly couples anatomically symmetric compartments:
       - Pair A: Medial Meniscus (idx 2) <-> Lateral Meniscus (idx 3)
       - Pair B: Medial OA (idx 4) <-> Lateral OA (idx 5)
       Computes bilateral symmetry contrast and cross-compartmental modulation.
    2. Global Multi-Head Finding Self-Attention:
       Full (12 x 12) graph interaction across all knee findings with clinical prior bias:
       - Unhappy Triad (ACL <-> Medial Meniscus <-> Bone Contusion)
       - Degenerative Complex (Medial OA <-> Baker's Cyst <-> Effusion)
    3. Zero-Initialized Residual Gate (gamma = 0.0):
       Guarantees 100% baseline identity match at Step 0, smoothly learning optimal relation infusion.
    """
    def __init__(self, dim=768, num_classes=12, num_heads=4, dropout=0.10, gamma_init=0.0):
        super().__init__()
        self.dim = dim
        self.num_classes = num_classes

        # ── 1. Bilateral Compartmental Cross-Gating ────────────────────────
        # Medial <-> Lateral pairs: (2, 3) for Meniscus, (4, 5) for OA
        self.bilateral_pairs = [(2, 3), (4, 5)]
        self.bilateral_gate = nn.Sequential(
            nn.Linear(dim * 2, dim // 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim // 2, dim * 2),
            nn.Sigmoid()
        )

        # ── 2. Global Cross-Finding Self-Attention ─────────────────────────
        self.norm = nn.LayerNorm(dim)
        self.cross_finding_attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        # Learnable Clinical Relation Prior Matrix (12, 12)
        prior_matrix = torch.zeros(num_classes, num_classes)
        # Bilateral / Symmetrical pairs
        prior_matrix[2, 3] = prior_matrix[3, 2] = 0.5   # Medial <-> Lateral Meniscus
        prior_matrix[4, 5] = prior_matrix[5, 4] = 0.5   # Medial <-> Lateral OA
        # Medial compartment complex
        prior_matrix[1, 2] = prior_matrix[2, 1] = 0.5   # MCL <-> Medial Meniscus
        prior_matrix[2, 4] = prior_matrix[4, 2] = 0.5   # Medial Meniscus <-> Medial OA
        # Degenerative / Fluid complex
        prior_matrix[4, 9] = prior_matrix[9, 4] = 0.5   # Medial OA <-> Baker's Cyst
        prior_matrix[7, 8] = prior_matrix[8, 7] = 0.5   # Effusion <-> Synovitis
        # Trauma complex (Unhappy Triad)
        prior_matrix[0, 2] = prior_matrix[2, 0] = 0.5   # ACL <-> Medial Meniscus
        prior_matrix[0, 10] = prior_matrix[10, 0] = 0.5 # ACL <-> Contusion
        prior_matrix[10, 11] = prior_matrix[11, 10] = 0.5 # Contusion <-> Fracture

        self.register_buffer("relation_prior", prior_matrix)
        self.prior_scale = nn.Parameter(torch.tensor(0.5, dtype=torch.float32))

        # Feed-forward refinement
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim)
        )

        # ── 3. Zero-Initialized Residual Gate ──────────────────────────────
        self.gamma = nn.Parameter(torch.full((1,), float(gamma_init)))

    def forward(self, z):
        """
        Input:  z of shape (B, 12, dim) - 12 finding representation tokens
        Output: z_refined of shape (B, 12, dim)
        """
        B, N, D = z.shape

        # 1. Bilateral Compartmental Coupling
        z_bi = z.clone()
        for idx_m, idx_l in self.bilateral_pairs:
            f_m = z[:, idx_m, :]
            f_l = z[:, idx_l, :]
            f_cat = torch.cat([f_m, f_l], dim=-1)
            gates = self.bilateral_gate(f_cat)
            gate_m, gate_l = gates.chunk(2, dim=-1)
            z_bi[:, idx_m, :] = f_m + gate_m * f_l
            z_bi[:, idx_l, :] = f_l + gate_l * f_m

        # 2. Multi-Head Cross-Finding Self-Attention
        h = self.norm(z_bi)
        attn_bias = self.prior_scale * self.relation_prior
        attn_out, _ = self.cross_finding_attn(h, h, h, attn_mask=attn_bias)

        # 3. Residual FFN
        h2 = self.norm2(z_bi + attn_out)
        out = h2 + self.ffn(h2)

        # 4. Zero-Init Residual Blending
        return z + self.gamma * (out - z)

class FindingSpecificAttentionHead(nn.Module):
    def __init__(self, dim=768, n_out=12, hidden=256, dropout=CFG.dropout,
                 use_prior=CFG.use_slot_prior, prior_strength=CFG.slot_prior_strength,
                 prior_sigma=CFG.slot_prior_sigma,
                 use_cosine=getattr(CFG, "use_cosine_classifier", True),
                 use_dual_pooling=getattr(CFG, "use_dual_pooling", True),
                 focal_temperature=getattr(CFG, "focal_temperature", 0.25),
                 use_plane_decoupled_pooling=getattr(CFG, "use_plane_decoupled_pooling", True),
                 use_finding_relation=getattr(CFG, "use_finding_relation", False),
                 use_dynamic_contrast_gating=getattr(CFG, "use_dynamic_contrast_gating", False)):
        super().__init__()
        self.dim = dim
        self.n_out = n_out
        self.use_prior = use_prior
        self.use_cosine = use_cosine
        self.use_dual_pooling = use_dual_pooling
        self.focal_temperature = focal_temperature
        self.broad_temperature = getattr(CFG, 'broad_temperature', getattr(CFG, 'focal_temperature', 2.0))
        self.use_plane_decoupled_pooling = use_plane_decoupled_pooling
        self.use_finding_relation = use_finding_relation
        if self.use_finding_relation:
            self.relation_head = BilateralFindingRelationHead(dim=dim, num_classes=n_out, num_heads=getattr(CFG, "relation_num_heads", 4), dropout=getattr(CFG, "relation_dropout", 0.10), gamma_init=getattr(CFG, "relation_gamma_init", 0.0))
        else:
            self.relation_head = None
        self.use_dynamic_contrast_gating = use_dynamic_contrast_gating

        # 🌟 Contrast-Aware Dynamic Inter-Plane Gating (4 contrast types -> 12 pathology gate shifts)
        if use_dynamic_contrast_gating:
            self.contrast_plane_gate = nn.Linear(getattr(CFG, 'n_contrast_types', 4), n_out, bias=False)
            nn.init.zeros_(self.contrast_plane_gate.weight)

        # 🌟 Inter-Plane Gating Residual Parameter (zero-initialized)
        self.plane_gate = nn.Parameter(torch.zeros(n_out, 3, dtype=torch.float32))

        self.norm = nn.LayerNorm(dim)
        self.att = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_out)
        )
        self.clsW = nn.Parameter(torch.zeros(n_out, dim))
        self.clsb = nn.Parameter(torch.zeros(n_out))
        nn.init.trunc_normal_(self.clsW, std=0.02)

        if use_cosine:
            # Learnable Temperature Scale initialized to log(16.0)
            self.logit_scale = nn.Parameter(torch.tensor(2.7725887, dtype=torch.float32)) # ln(16.0)

        # 🌟 Dual-Pooling: Pathology-specific zero-initialized residual focal gate
        if use_dual_pooling:
            self.focal_gate = nn.Parameter(torch.zeros(n_out, 1, dtype=torch.float32))
        else:
            self.focal_gate = None

        if use_prior:
            s_per_plane = getattr(CFG, "num_slices_per_plane", 7)
            self.learnable_slot_prior = WarmStartResidualGaussianSlotPrior(n_classes=n_out, n_slices_per_plane=s_per_plane)
            init_val = float(np.log(np.exp(prior_strength) - 1.0))
            self.prior_scale = nn.Parameter(torch.ones(1, 1, n_out, dtype=torch.float32) * init_val)
        else:
            self.learnable_slot_prior = None
            self.prior_scale = None

    def forward(self, feats, plane_mask=None, c_plane_dist=None):
        h = self.norm(feats)                            # (B, 21, 768)
        a_logits = self.att(h)                          # (B, 21, 12)
        if self.use_prior and self.learnable_slot_prior is not None:
            effective_scale = F.softplus(self.prior_scale)
            slot_prior = self.learnable_slot_prior()
            a_logits = a_logits + effective_scale * slot_prior.unsqueeze(0)

        if plane_mask is not None:
            if plane_mask.dim() == 2:
                plane_mask = plane_mask.unsqueeze(-1)
            a_logits = a_logits.masked_fill(plane_mask, -1e4)

        if not getattr(self, "use_plane_decoupled_pooling", False):
            # Monolithic 21-element Baseline
            a_ctx = torch.softmax(a_logits, dim=1)          # (B, 21, 12)
            pooled = torch.einsum("bsn, bsf -> bnf", a_ctx, h) # (B, 12, 768)

            if self.use_dual_pooling and self.focal_gate is not None:
                # 🌟 Dual-Scale Context: soft smoothed attention across all slices (tau=2.0)
                tau_broad = max(getattr(self, "broad_temperature", getattr(self, "focal_temperature", 2.0)), 1e-3)
                a_broad = torch.softmax(a_logits / tau_broad, dim=1) # (B, 21, 12)
                pooled_broad = torch.einsum("bsn, bsf -> bnf", a_broad, h) # (B, 12, 768)
                #  Linear Focal Gating (Zero-Init: Starts at 100% Focal, smoothly learns Broad context without 50% prior bias)
                pooled = pooled + self.focal_gate * (pooled_broad - pooled)
        else:
            # 🌟 Plane-Decoupled Hierarchical Dual-Pooling (Intra-Plane + Inter-Plane Gating)
            B, S, F_dim = h.shape
            P = 3
            S_p = S // P  # 7 slices per plane
            h_3d = h.view(B, P, S_p, F_dim)
            a_logits_3d = a_logits.view(B, P, S_p, self.n_out)

            # Level 1: Intra-Plane Attention Pooling (dim=2: softmax over 7 slices within each plane)
            a_ctx_intra = torch.softmax(a_logits_3d, dim=2) # (B, P=3, S_p=7, N=12)
            z_ctx = torch.einsum("bpsn, bpsf -> bpnf", a_ctx_intra, h_3d) # (B, P=3, N=12, F=768)

            if self.use_dual_pooling and self.focal_gate is not None:
                # 🌟 Dual-Scale Context: soft smoothed attention across all slices (tau=2.0)
                tau_broad = max(getattr(self, "broad_temperature", getattr(self, "focal_temperature", 2.0)), 1e-3)
                a_broad_intra = torch.softmax(a_logits_3d / tau_broad, dim=2) # (B, P=3, S_p=7, N=12)
                z_broad = torch.einsum("bpsn, bpsf -> bpnf", a_broad_intra, h_3d) # (B, P=3, N=12, F=768)
                #  Linear Focal Gating (Zero-Init: Starts at 100% Focal, smoothly learns Broad context without 50% prior bias)
                z_intra = z_ctx + self.focal_gate.view(1, 1, self.n_out, 1) * (z_broad - z_ctx)
            else:
                z_intra = z_ctx

            # Level 2: Inter-Plane Gating
            if self.use_prior and self.learnable_slot_prior is not None:
                _, _, pi = self.learnable_slot_prior.get_geometry() # (12, 3)
                w_logits = pi + self.plane_gate                     # (12, 3)
            else:
                w_logits = self.plane_gate                          # (12, 3)

            w_logits = w_logits.unsqueeze(0).expand(B, -1, -1)     # (B, 12, 3)

            # 🌟 Dynamic Contrast-Aware Inter-Plane Shift
            if getattr(self, 'use_dynamic_contrast_gating', False) and c_plane_dist is not None:
                # c_plane_dist: (B, 3, 4) -> map each plane's contrast to 12 disease affinities -> (B, 3, 12) -> (B, 12, 3)
                delta_w = self.contrast_plane_gate(c_plane_dist).permute(0, 2, 1)
                gamma = getattr(CFG, 'dynamic_plane_gamma', 1.0)
                w_logits = w_logits + gamma * delta_w

            # Mask out dropped plane if Whole-Plane Dropout occurred
            if plane_mask is not None:
                dropped_planes = plane_mask.view(B, P, S_p).all(dim=-1) # (B, 3)
                if dropped_planes.any():
                    w_logits = w_logits.masked_fill(dropped_planes.unsqueeze(1), -1e4)

            w_plane = torch.softmax(w_logits, dim=-1) # (B, 12, 3) - sum over 3 planes = 1.0
            pooled = torch.einsum("bpnf, bnp -> bnf", z_intra, w_plane) # (B, 12, 768)

        # 🌟 Bilateral Finding Relation Head
        if getattr(self, "use_finding_relation", False) and self.relation_head is not None:
            pooled = self.relation_head(pooled)

        if self.use_cosine:
            pooled_norm = F.normalize(pooled, p=2, dim=-1, eps=1e-6)   # (B, 12, 768)
            clsW_norm   = F.normalize(self.clsW,   p=2, dim=-1, eps=1e-6)   # (12, 768)
            scale = torch.exp(torch.clamp(self.logit_scale, max=4.158883))               # max scale = 64.0
            cos_sim = (pooled_norm * clsW_norm.unsqueeze(0)).sum(-1)    # (B, 12)
            logits  = scale * cos_sim + self.clsb
        else:
            logits = (pooled * self.clsW).sum(-1) + self.clsb          # (B, 12)

        return logits


class ConvNeXt25DModel(nn.Module):
    def __init__(self, model_name=CFG.backbone, num_classes=12, unfreeze_last=CFG.unfreeze_last,
                 head_type=CFG.head_type, use_plane_embedding=CFG.use_plane_embedding,
                 use_contrast_embedding=CFG.use_contrast_embedding,
                 use_contrast_aux=CFG.use_contrast_aux,
                 n_contrast=CFG.n_contrast_types,
                 use_cross_plane=getattr(CFG, "use_cross_plane", False),
                 use_plane_dropout=CFG.use_plane_dropout,
                 plane_dropout_prob=CFG.plane_dropout_prob,
                 use_metadata_injection=getattr(CFG, "use_metadata_injection", True),
                 meta_dim=getattr(CFG, "meta_dim", 13)):
        super().__init__()
        self.backbone = load_convnext_backbone(model_name, drop_path_rate=getattr(CFG, "drop_path_rate", 0.08))
        feat_dim = self.backbone.embed_dim      # 768 dims
        self.use_plane_embedding = use_plane_embedding
        self.use_contrast_embedding = use_contrast_embedding
        self.use_contrast_aux = use_contrast_aux
        self.n_contrast = n_contrast
        self.use_cross_plane = use_cross_plane
        self.use_intra_plane_depth = getattr(CFG, "use_intra_plane_depth", False)
        if self.use_intra_plane_depth:
            self.depth_block = IntraPlaneVolumetricDepthBlock(dim=feat_dim, n_planes=3, dropout=getattr(CFG, "depth_interact_dropout", 0.10))
        else:
            self.depth_block = None
        self.use_plane_dropout = use_plane_dropout
        self.plane_dropout_prob = plane_dropout_prob
        self.use_metadata_injection = use_metadata_injection

        if use_plane_embedding:
            self.plane_emb = nn.Parameter(torch.randn(3, feat_dim) * 0.02)
            s_p = getattr(CFG, 'num_slices_per_plane', 7)
            plane_idx = torch.tensor([0]*s_p + [1]*s_p + [2]*s_p, dtype=torch.long)
            self.register_buffer("plane_idx", plane_idx)

        if use_contrast_embedding:
            self.contrast_emb = nn.Embedding(n_contrast, feat_dim)
            nn.init.trunc_normal_(self.contrast_emb.weight, std=0.02)

        # 🌟 Auxiliary Contrast Prediction Head
        if use_contrast_aux:
            self.contrast_head = nn.Sequential(
                nn.LayerNorm(feat_dim),
                nn.Linear(feat_dim, 64),
                nn.GELU(),
                nn.Dropout(0.10),
                nn.Linear(64, n_contrast)
            )

        # 🌟 13-Dim Tabular Patient & Scanner Metadata Injection Modules
        if self.use_metadata_injection:
            self.use_meta_film = getattr(CFG, "use_meta_film", False)
            if self.use_meta_film:
                self.meta_encoder = nn.Sequential(
                    nn.Linear(meta_dim, 128),
                    nn.LayerNorm(128),
                    nn.SiLU(),
                    nn.Dropout(getattr(CFG, "meta_mlp_dropout", 0.20)),
                    nn.Linear(128, feat_dim)
                )
                self.meta_gamma = nn.Parameter(torch.zeros(1))  # Zero-init gating for smooth warm-up
            else:
                self.meta_encoder = None
                self.meta_gamma = None

            self.meta_finding_bias = nn.Sequential(
                nn.Linear(meta_dim, 64),
                nn.LayerNorm(64),
                nn.SiLU(),
                nn.Dropout(getattr(CFG, "meta_mlp_dropout", 0.20)),
                nn.Linear(64, num_classes)
            )
        else:
            self.use_meta_film = False
            self.meta_encoder = None
            self.meta_gamma = None
            self.meta_finding_bias = None

        if use_cross_plane:
            self.cross_plane = CrossPlaneFusionBlock(dim=feat_dim, num_heads=CFG.cross_plane_heads, dropout=CFG.cross_plane_dropout)
        else:
            self.cross_plane = None

        self.head = FindingSpecificAttentionHead(dim=feat_dim, n_out=num_classes)

        # Freeze early stages, unfreeze top stages
        for param in self.backbone.parameters():
            param.requires_grad = False
        bb_module = self.backbone.model if hasattr(self.backbone, "model") else self.backbone

        # timm ConvNeXt structure: stages[0], stages[1], stages[2], stages[3]
        if hasattr(bb_module, "stages"):
            for stage in bb_module.stages[-unfreeze_last:]:
                for param in stage.parameters():
                    param.requires_grad = True
            if hasattr(bb_module, "norm_pre"):
                for param in bb_module.norm_pre.parameters(): param.requires_grad = True
            if hasattr(bb_module, "head"):
                for param in bb_module.head.parameters(): param.requires_grad = True
        else:
            for param in list(bb_module.parameters())[-30:]:
                param.requires_grad = True

        # 🌟 Explicitly Unfreeze GeM Pooling & Multi-Stage Feature Fusion in Backbone Wrapper!
        if hasattr(self.backbone, "pool") and self.backbone.pool is not None:
            for param in self.backbone.pool.parameters():
                param.requires_grad = True
        if hasattr(self.backbone, "pool_stage3") and self.backbone.pool_stage3 is not None:
            for param in self.backbone.pool_stage3.parameters():
                param.requires_grad = True
        if hasattr(self.backbone, "stage3_proj") and self.backbone.stage3_proj is not None:
            for param in self.backbone.stage3_proj.parameters():
                param.requires_grad = True
        if hasattr(self.backbone, "stage3_gamma") and self.backbone.stage3_gamma is not None:
            self.backbone.stage3_gamma.requires_grad = True
        if hasattr(self.backbone, 'fpn') and self.backbone.fpn is not None:
            for param in self.backbone.fpn.parameters():
                param.requires_grad = True

    def forward(self, x, contrast_flags=None, meta_vec=None):
        B, S, C, H, W = x.shape
        x_flat = x.view(B * S, C, H, W)
        if getattr(CFG, "use_channels_last", True):
            x_flat = x_flat.contiguous(memory_format=torch.channels_last)
        chunk_size = getattr(CFG, "backbone_chunk_size", 672)
        if x_flat.size(0) > chunk_size and self.training:
            raw_feats = torch.cat([self.backbone(c) for c in x_flat.split(chunk_size)], dim=0)
        else:
            raw_feats = self.backbone(x_flat)
        raw_feats = raw_feats.view(B, S, -1)                # (B, S, 768)

        aux_logits = None
        if self.use_contrast_aux:
            aux_logits = self.contrast_head(raw_feats)       # (B, 15, 4)

        feats = raw_feats

        if self.use_plane_embedding:
            feats = feats + self.plane_emb[self.plane_idx].unsqueeze(0)

        if self.use_contrast_embedding and contrast_flags is not None:
            c_clamped = contrast_flags.clamp(0, self.n_contrast - 1)
            feats = feats + self.contrast_emb(c_clamped)

        # 🌟 Feature-Level Metadata FiLM Injection with Modality Dropout
        meta_drop_mask = None
        if self.use_metadata_injection and meta_vec is not None:
            if self.training and getattr(CFG, "meta_dropout_prob", 0.0) > 0:
                # Modality Dropout: 25% of train samples get 0 metadata contribution, forcing 100% reliance on visual features
                meta_drop_mask = (torch.rand(meta_vec.size(0), 1, device=meta_vec.device) > CFG.meta_dropout_prob).float()

            if getattr(self, "use_meta_film", False) and self.meta_encoder is not None:
                meta_feat = self.meta_encoder(meta_vec).unsqueeze(1) # (B, 1, 768)
                if meta_drop_mask is not None:
                    meta_feat = meta_feat * meta_drop_mask.unsqueeze(1) # (B, 1, 1)
                feats = feats + self.meta_gamma * meta_feat

        # 🛡️ EXP 3.2: Pre-Fusion Whole-Plane Dropout (Zero Information Leakage)
        plane_mask = None
        if self.training and self.use_plane_dropout and self.plane_dropout_prob > 0:
            mask_tensor = torch.zeros((B, CFG.num_slices, 1), dtype=torch.bool, device=feats.device)
            s_per_plane = getattr(CFG, "num_slices_per_plane", 7)
            for b_idx in range(B):
                if random.random() < self.plane_dropout_prob:
                    drop_plane = random.randint(0, 2)
                    start_s = drop_plane * s_per_plane
                    end_s   = start_s + s_per_plane
                    mask_tensor[b_idx, start_s:end_s, :] = True
            if mask_tensor.any():
                plane_mask = mask_tensor
                # Zero out dropped plane features BEFORE CrossPlaneFusion to prevent leakage across attention!
                feats = feats.masked_fill(plane_mask, 0.0)

        if self.use_cross_plane and self.cross_plane is not None:
            feats = self.cross_plane(feats)

        # 🌟 Compute Plane-Specific Contrast Distribution (B, 3, 4) for Dynamic Inter-Plane Gating
        c_plane_dist = None
        if self.use_contrast_aux and aux_logits is not None:
            s_p = getattr(CFG, 'num_slices_per_plane', S // 3)
            c_plane_dist = torch.softmax(aux_logits.view(B, 3, s_p, self.n_contrast), dim=-1).mean(dim=2) # (B, 3, 4)

        #  Intra-Plane 3D Volumetric Depth Interaction (Before Hierarchical Attention Pooling)
        if self.use_intra_plane_depth and self.depth_block is not None:
            feats = self.depth_block(feats)

        main_logits = self.head(feats, plane_mask=plane_mask, c_plane_dist=c_plane_dist)  # (B, 12)

        # 🌟 Logit-Level Clinical Prior Bias Injection (with Modality Dropout + Reduced Weight)
        if self.use_metadata_injection and meta_vec is not None:
            bias = self.meta_finding_bias(meta_vec)
            if meta_drop_mask is not None:
                bias = bias * meta_drop_mask
            bias_w = getattr(CFG, "meta_logit_bias_weight", 0.03)
            main_logits = main_logits + bias_w * bias

        if self.use_contrast_aux:
            return main_logits, aux_logits
        return main_logits

print(f"✅ ConvNeXt-Tiny Model initialized with Cosine Classifier, Metadata ({CFG.meta_dim}-dim), Aux Contrast & Contrast-Aware Dynamic Plane Gating.")




✅ ConvNeXt-Tiny Model initialized with Cosine Classifier, Metadata (13-dim), Aux Contrast & Contrast-Aware Dynamic Plane Gating.


---
## 5. Class-Calibrated Asymmetric Loss (ASL), Soft-Margin Rank Loss, 2.5D Mixup & EMA


In [7]:
class Ema:
    def __init__(self, model, decay=CFG.ema_decay):
        self.decay = decay
        self.step  = 0
        self.model = None
        if decay > 0:
            self.model = copy.deepcopy(model).eval()
            for p in self.model.parameters():
                p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        if self.model is None:
            return
        self.step += 1
        decay = min(self.decay, (1.0 + self.step) / (10.0 + self.step))
        # ⚡ High-speed fused in-place lerp_ (zero temporary tensor allocations)
        for p_ema, p_model in zip(self.model.parameters(), model.parameters()):
            p_ema.lerp_(p_model, 1.0 - decay)
        for b_ema, b_model in zip(self.model.buffers(), model.buffers()):
            if b_ema.is_floating_point():
                b_ema.lerp_(b_model, 1.0 - decay)

    def target(self, model):
        return self.model if self.model is not None else model

# ─── 🌟 EXP 1.2: Contrast-Aware & Metadata-Aware 2.5D Mixup ─────────────────
def mixup_data_25d(imgs, labels, weights, is_gold, c_flags=None, meta_vec=None, alpha=CFG.mixup_alpha, prob=CFG.mixup_prob):
    # Convert c_flags to one-hot float distributions (B, S, 4) for smooth continuous mixup
    n_contrast = getattr(CFG, "n_contrast_types", 4)
    if c_flags is not None:
        c_onehot = F.one_hot(c_flags.clamp(0, n_contrast - 1), num_classes=n_contrast).float()
    else:
        c_onehot = None

    if alpha > 0 and random.random() < prob and imgs.size(0) > 1:
        lam = np.random.beta(alpha, alpha)
        batch_size = imgs.size(0)
        index = torch.randperm(batch_size, device=imgs.device)

        mixed_imgs = lam * imgs + (1.0 - lam) * imgs[index]
        mixed_labels = lam * labels + (1.0 - lam) * labels[index]
        mixed_weights = lam * weights + (1.0 - lam) * weights[index]
        #  Gold-Preserving Mixup: Mẫu hỗn hợp vẫn được hưởng trọng số Gold (3.0x) nếu có sự tham gia của ca Gold
        mixed_is_gold = torch.clamp(is_gold + is_gold[index], 0.0, 1.0)
        mixed_c_dist = lam * c_onehot + (1.0 - lam) * c_onehot[index] if c_onehot is not None else None
        mixed_meta = lam * meta_vec + (1.0 - lam) * meta_vec[index] if meta_vec is not None else None
        return mixed_imgs, mixed_labels, mixed_weights, mixed_is_gold, mixed_c_dist, mixed_meta, True

    return imgs, labels, weights, is_gold, c_onehot, meta_vec, False


class EarlyStopping:
    def __init__(self, patience=CFG.early_stopping_patience, min_delta=CFG.early_stopping_min_delta,
                 min_epoch=getattr(CFG, "min_epoch_early_stop", 6), mode="max", verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.min_epoch = min_epoch
        self.mode = mode
        self.verbose = verbose
        self.best_score = -np.inf if mode == "max" else np.inf
        self.counter = 0
        self.early_stop = False

    def step(self, epoch, score):
        improved = (score > self.best_score + self.min_delta) if self.mode == "max" else (score < self.best_score - self.min_delta)
        if improved:
            self.best_score = score
            self.counter = 0
            return True
        else:
            # Guard against premature early stopping during OneCycleLR warmup epochs
            if epoch >= self.min_epoch:
                self.counter += 1
                if self.verbose:
                    print(f"   ⏳ EarlyStopping: {self.counter}/{self.patience} (Best AUC: {self.best_score:.4f})")
                if self.counter >= self.patience:
                    self.early_stop = True
                    if self.verbose:
                        print("   🛑 Early Stopping triggered! Stopping training for this fold.")
            else:
                if self.verbose:
                    print(f"   ℹ️ Epoch {epoch} < min_epoch_early_stop ({self.min_epoch}): EarlyStopping counter suppressed.")
            return False

def stable_asymmetric_loss(logits, targets, weights, is_gold_tensor,
                           gamma_pos=CFG.asl_gamma_pos, gamma_neg=CFG.asl_gamma_neg,
                           asl_clip=getattr(CFG, "asl_clip", 0.05),
                           pu_weights=None, fn_weights=None):
    """
    ⚔️ SOTA Asymmetric Loss (ASL, Ridnik et al., ICCV 2021) with Negative Margin Clipping:
    - Clips easy negative probabilities below asl_clip (m=0.05) to 0, completely zeroing out
      easy background negative gradient noise across multi-label classes.
    - Preserves unsuppressed positive gradients (gamma_pos=0.0).
    - Fully integrates Continuous False Negative (FN) scaling and Class-Calibrated Soft PU Loss.
    """
    if pu_weights is None:
        pu_weights = PU_WEIGHTS_TENSOR
    if fn_weights is None:
        fn_weights = FN_WEIGHTS_TENSOR
    t = torch.nan_to_num(targets, nan=0.5)
    p = torch.sigmoid(logits)

    # 1. Asymmetric Probability Shift (Margin Clipping for Negatives)
    p_neg = (p - asl_clip).clamp(min=0.0) if (asl_clip is not None and asl_clip > 0) else p

    # 2. Asymmetric Cross Entropy Losses (Safe eps-clamping against log(0))
    eps = 1e-8
    loss_pos = -torch.log(p.clamp(min=eps))
    loss_neg = -torch.log((1.0 - p_neg).clamp(min=eps))
    bce_asym = t * loss_pos + (1.0 - t) * loss_neg

    with torch.no_grad():
        # 3. Asymmetric Focal Weighting
        w_pos = torch.pow((1.0 - p).clamp(min=0.0, max=1.0), gamma_pos)
        w_neg = torch.pow(p_neg.clamp(min=0.0, max=1.0), gamma_neg)
        asym_weight = t * w_pos + (1.0 - t) * w_neg

        # 4. Zero-copy PU & FN Weighting
        fn_w = fn_weights.to(logits.device) if fn_weights.device != logits.device else fn_weights
        pu_w = pu_weights.to(logits.device) if pu_weights.device != logits.device else pu_weights
        fn_weight = 1.0 + (fn_w - 1.0) * t
        pu_scale = t + (1.0 - t) * pu_w

    loss_noisy = bce_asym * asym_weight * fn_weight * pu_scale
    loss_gold  = bce_asym * asym_weight

    gold_mask = is_gold_tensor.unsqueeze(1).float()
    loss_mat = gold_mask * loss_gold + (1.0 - gold_mask) * loss_noisy
    return (loss_mat * weights).mean()

def rank_loss(logits, targets, weights):
    parts = []
    usable = weights > 0.3
    for j in range(logits.shape[1]):
        t_col = targets[:, j]
        w_col = usable[:, j]
        pos_mask = (t_col > 0.55) & w_col
        neg_mask = (t_col < 0.45) & w_col
        pos_logits = logits[pos_mask, j]
        neg_logits = logits[neg_mask, j]
        pos_targets = t_col[pos_mask]
        neg_targets = t_col[neg_mask]
        if len(pos_logits) > 0 and len(neg_logits) > 0:
            delta_t = (pos_targets[:, None] - neg_targets[None, :]).clamp(min=0.1)
            diff = torch.clamp(pos_logits[:, None] - neg_logits[None, :], min=-12.0, max=12.0)
            parts.append((delta_t * F.softplus(-diff)).mean())
    return torch.stack(parts).mean() if parts else logits.new_tensor(0.0)

def compute_loss(logits, targets, weights, is_gold_tensor, pu_weights=None, fn_weights=None):
    asl = stable_asymmetric_loss(logits, targets, weights, is_gold_tensor, pu_weights=pu_weights, fn_weights=fn_weights)
    rloss = rank_loss(logits, targets, weights)
    return asl + CFG.rank_loss_w * rloss

def compute_macro_auc(y_true, y_pred, weights=None):
    aucs = []
    for i in range(y_true.shape[1]):
        t, p = y_true[:, i], y_pred[:, i]
        if weights is not None:
            w = weights[:, i]
            valid = (w > 0.3) & (~np.isnan(t))
        else:
            valid = ~np.isnan(t)

        tv, pv = t[valid], p[valid]
        tv_bin = (tv > 0.5).astype(np.int32)
        # 🌟 Bỏ qua class chỉ có 1 nhãn (ví dụ 100% âm tính trong holdout nhỏ), tuyệt đối không gán 0.5 gây nhiễu
        if len(np.unique(tv_bin)) > 1:
            try:
                score = roc_auc_score(tv_bin, pv)
                if not np.isnan(score):
                    aucs.append(score)
            except Exception:
                pass
    return float(np.mean(aucs)) if aucs else 0.5

print("✅ SOTA ASL (Negative Margin Clipping m=0.05) + Soft PU Loss + Metadata-Aware Mixup ready.")


# ── ⚡ GPU Batch Augmentation & Normalization Engine (< 3ms per batch) ──────
# 🌟 Uniform Grayscale MRI Stats for 2.5D Triplets (Eliminates artificial slice offset)
GPU_IMAGENET_MEAN = torch.tensor([0.450, 0.450, 0.450], device=CFG.device).view(1, 1, 3, 1, 1)
GPU_IMAGENET_STD  = torch.tensor([0.225, 0.225, 0.225], device=CFG.device).view(1, 1, 3, 1, 1)

def gpu_augment_batch(imgs_u8, labels, weights, is_gold, c_flags, meta_t, is_right=None):
    """
    Performs Laterality Normalization, Depth Jitter, Gamma, Rotation, Brightness, Mixup, and Normalization
    fully parallelized on CUDA GPU in ~3 ms for batch size 16.
    """
    B, S, C, H, W = imgs_u8.shape
    imgs = imgs_u8.float() / 255.0  # (B, 21, 3, 224, 224) on GPU

    # 0. 🏥 Vectorized GPU Laterality Normalization (< 0.05ms for entire batch on CUDA)
    if CFG.use_laterality_flip and is_right is not None and is_right.any():
        s_plane = getattr(CFG, "num_slices_per_plane", 7)
        imgs[is_right, s_plane:s_plane*3] = torch.flip(imgs[is_right, s_plane:s_plane*3], dims=[-1])
        imgs[is_right, 0:s_plane] = torch.flip(imgs[is_right, 0:s_plane], dims=[1, 2])

    # 1. 🌪️ 3D Depth Jittering (GPU Vectorized)
    if CFG.use_depth_jitter and random.random() < CFG.depth_jitter_prob:
        s_plane = getattr(CFG, "num_slices_per_plane", 7)
        for p_idx in range(3):
            start_idx = p_idx * s_plane
            delta = random.uniform(-CFG.depth_jitter_max_shift, CFG.depth_jitter_max_shift)
            if abs(delta) > 1e-4:
                plane_slices = imgs[:, start_idx:start_idx + s_plane]
                target = torch.cat([plane_slices[:, 1:], plane_slices[:, -1:]], dim=1) if delta > 0 else torch.cat([plane_slices[:, :1], plane_slices[:, :-1]], dim=1)
                alpha = abs(delta)
                imgs[:, start_idx:start_idx + s_plane] = (1.0 - alpha) * plane_slices + alpha * target

    # 2. 🩻 2.5D Channel Slice Drop (GPU Vectorized)
    if getattr(CFG, "use_slice_drop", True) and random.random() < getattr(CFG, "slice_drop_prob", 0.20):
        if random.random() < 0.50:
            imgs[:, :, 0, :, :] = imgs[:, :, 1, :, :]  # Replace prior slice with central slice
        if random.random() < 0.50:
            imgs[:, :, 2, :, :] = imgs[:, :, 1, :, :]  # Replace next slice with central slice

    # 3. 🩻 MRI Physics: Random Gamma Power-Law Simulation (Vectorized Per-Sample Tensor)
    if getattr(CFG, "use_gamma_aug", True) and random.random() < getattr(CFG, "gamma_prob", 0.40):
        gamma = torch.empty((B, 1, 1, 1, 1), device=imgs.device).uniform_(
            getattr(CFG, "gamma_lo", 0.85), getattr(CFG, "gamma_hi", 1.20)
        )
        imgs = torch.pow(imgs.clamp(min=1e-5, max=1.0), gamma)


    # 4. 🔄 2.5D Ultra-Fast Vectorized GPU Affine (Fused CUDA Kernel, 5x faster than TF.affine)
    if getattr(CFG, "use_affine_aug", True) and random.random() < getattr(CFG, "affine_prob", 0.60):
        angle = random.uniform(-getattr(CFG, "affine_max_rotate", 6.0), getattr(CFG, "affine_max_rotate", 6.0))
        rad = -math.radians(angle)
        scale_lo, scale_hi = getattr(CFG, "affine_scale_range", (0.95, 1.05))
        scale = random.uniform(scale_lo, scale_hi)
        inv_s = 1.0 / max(scale, 1e-4)
        max_t = getattr(CFG, "affine_max_translate", 0.03) * 2.0
        tx = random.uniform(-max_t, max_t)
        ty = random.uniform(-max_t, max_t)
        theta = torch.tensor([
            [inv_s * math.cos(rad), -inv_s * math.sin(rad), -tx],
            [inv_s * math.sin(rad),  inv_s * math.cos(rad), -ty]
        ], device=imgs.device, dtype=imgs.dtype).unsqueeze(0).expand(B * S, -1, -1)
        grid = F.affine_grid(theta, [B * S, C, H, W], align_corners=False)
        imgs = F.grid_sample(imgs.view(B * S, C, H, W), grid, mode='bilinear', padding_mode='reflection', align_corners=False).view(B, S, C, H, W)

    # 5. 💡 Random Brightness (Vectorized Per-Sample Tensor)
    if random.random() < 0.30:
        bright = torch.empty((B, 1, 1, 1, 1), device=imgs.device).uniform_(0.90, 1.10)
        imgs = torch.clamp(imgs * bright, 0.0, 1.0)

    # 6. 🔀 2.5D Multi-Plane Mixup (GPU Vectorized)
    imgs, labels, weights, is_gold, c_flags, meta_t, is_mixed = mixup_data_25d(imgs, labels, weights, is_gold, c_flags, meta_t)

    # 7. 📐 ImageNet Normalization (GPU Vectorized)
    imgs = (imgs - GPU_IMAGENET_MEAN) / GPU_IMAGENET_STD
    return imgs, labels, weights, is_gold, c_flags, meta_t, is_mixed

def gpu_normalize_val_batch(imgs_u8, is_right=None):
    """Ultra-fast GPU conversion, laterality normalization, and ImageNet normalization for Validation."""
    imgs = imgs_u8.float() / 255.0
    if CFG.use_laterality_flip and is_right is not None and is_right.any():
        s_plane = getattr(CFG, "num_slices_per_plane", 7)
        imgs[is_right, s_plane:s_plane*3] = torch.flip(imgs[is_right, s_plane:s_plane*3], dims=[-1])
        imgs[is_right, 0:s_plane] = torch.flip(imgs[is_right, 0:s_plane], dims=[1, 2])
    imgs = (imgs - GPU_IMAGENET_MEAN) / GPU_IMAGENET_STD
    return imgs


✅ SOTA ASL (Negative Margin Clipping m=0.05) + Soft PU Loss + Metadata-Aware Mixup ready.


---
## 6. Stage 1: Pure Visual Pretraining with Per-Finding Loss Masking


In [8]:
import os
import gc
import math
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

os.makedirs("./checkpoints", exist_ok=True)

# ==============================================================================
# 🛠️ 1. MODULAR STAGE TRAINING RUNNER (5-FOLD COMPATIBLE)
# ==============================================================================
def run_stage_training(stage_df, stage_num=1, epochs=CFG.epochs, ckpt_prefix="best_convnext_tiny_kaggle"):
    """
    Huấn luyện toàn diện K-Fold cho Stage 1 hoặc Stage 2.
    - Stage 1: Masking nhãn text mập mờ (w <= 0.30), bảo vệ 100% Gold Standard.
    - Stage 2: Huấn luyện trên nhãn tinh chế đã được Visual Model giải cứu.
    """
    oof_predictions = np.zeros((len(stage_df), len(TARGET_COLS)))
    oof_targets     = np.zeros((len(stage_df), len(TARGET_COLS)))
    oof_weights     = np.zeros((len(stage_df), len(TARGET_COLS)))
    fold_best_aucs  = []

    print(f"\n{'='*86}")
    print(f"🚀 STARTING STAGE {stage_num} TRAINING FOR FOLDS: {CFG.train_folds} ({epochs} Epochs OneCycleLR)")
    print(f"{'='*86}")

    for fold in CFG.train_folds:
        print(f"\n========================== STAGE {stage_num} — FOLD {fold} ==========================")
        train_df = stage_df[stage_df["fold"] != fold].reset_index(drop=True)
        val_df   = stage_df[stage_df["fold"] == fold].reset_index(drop=True)

        if stage_num == 1 and getattr(CFG, "stage1_confident_only", False):
            print(f"🎯 [STAGE 1 ACTIVE] Fold {fold} Train: Áp dụng Per-Finding Loss Masking (w <= {CFG.min_train_confidence}).")
            print(f"   🛡️ Val fold {fold} giữ nguyên 100% ({len(val_df)} ca) cho đánh giá khách quan.")
        else:
            print(f"🎯 [STAGE 2 ACTIVE] Fold {fold} Train: Huấn luyện trên nhãn tinh chế (Refined Soft Labels), không áp dụng masking.")

        # 🌟 100% Zero-Leakage Auto PU & FN Weights Calibration strictly on train_df
        fold_fn_list, fold_pu_list, FN_WEIGHTS_TENSOR, PU_WEIGHTS_TENSOR = compute_auto_pu_fn_weights_per_fold(
            train_df, TARGET_COLS, CONF_COLS, device=CFG.device
        )
        print(f"⚖️ Fold {fold} Auto-Calibrated PU Weights (Negative scale):")
        print("  ", {c: round(v, 3) for c, v in zip(TARGET_COLS, fold_pu_list)})
        print(f"⚖️ Fold {fold} Auto-Calibrated FN Weights (Positive boost):")
        print("  ", {c: round(v, 3) for c, v in zip(TARGET_COLS, fold_fn_list)})

        train_ds = KneeMRIDataset(train_df, mmap_data, lat_map=study_lat_map, contrast_map=study_contrast_map, meta_map=study_meta_map, is_train=True)
        val_ds   = KneeMRIDataset(val_df,   mmap_data, lat_map=study_lat_map, contrast_map=study_contrast_map, meta_map=study_meta_map, is_train=False)

        if stage_num == 1 and getattr(CFG, "stage1_confident_only", False):
            n_masked = (train_ds.precomputed_weights == 0.0).sum()
            print(f"🎯 [STAGE 1 PER-FINDING MASK] Triệt tiêu loss của {n_masked:,} vị trí nhãn mập mờ trên tập Train!")

        train_loader = DataLoader(
            train_ds,
            batch_size=CFG.batch_size,
            shuffle=True,
            num_workers=CFG.num_workers,
            pin_memory=CFG.pin_memory,
            prefetch_factor=getattr(CFG, "prefetch_factor", 2) if CFG.num_workers > 0 else None,
            persistent_workers=True if CFG.num_workers > 0 else False,
            drop_last=True
        )
        val_loader = DataLoader(
            val_ds,
            batch_size=getattr(CFG, "val_batch_size", 32),
            shuffle=False,
            num_workers=CFG.num_workers,
            pin_memory=CFG.pin_memory,
            prefetch_factor=getattr(CFG, "prefetch_factor", 2) if CFG.num_workers > 0 else None,
            persistent_workers=True if CFG.num_workers > 0 else False
        )

        model = ConvNeXt25DModel(model_name=CFG.backbone, num_classes=len(TARGET_COLS), unfreeze_last=CFG.unfreeze_last, head_type=CFG.head_type)
        model.to(CFG.device)
        if getattr(CFG, "use_channels_last", True):
            model.to(memory_format=torch.channels_last)
        ema = Ema(model, decay=CFG.ema_decay)
        print(f"⚡ Model Config: Dual-Pooling={CFG.use_dual_pooling} (Broad τ={getattr(CFG, 'broad_temperature', 2.0)}), FPN-Fusion={getattr(model.backbone, 'use_fpn_fusion', False)}, Slot-Prior={CFG.use_slot_prior}")

        bb_params       = [p for n, p in model.named_parameters() if n.startswith("backbone.") and "pool" not in n and "stage3_" not in n and "fpn" not in n and "gamma" not in n and p.requires_grad]
        fpn_params      = [p for n, p in model.named_parameters() if "fpn" in n and "pool" not in n and "gamma" not in n and p.requires_grad]
        stage3_params   = [p for n, p in model.named_parameters() if "stage3_" in n and "pool" not in n and "fpn" not in n and "gamma" not in n and p.requires_grad]
        gem_params      = [p for n, p in model.named_parameters() if "pool" in n and ".p" in n and p.requires_grad]
        meta_params     = [p for n, p in model.named_parameters() if "meta_" in n and "gamma" not in n and p.requires_grad]
        prior_params    = [p for n, p in model.named_parameters() if "learnable_slot_prior" in n and p.requires_grad]
        relation_params = [p for n, p in model.named_parameters() if "relation_head" in n and "gamma" not in n and p.requires_grad]
        gate_params     = [p for n, p in model.named_parameters() if ("gamma" in n or "gate" in n) and "learnable_slot_prior" not in n and p.requires_grad]
        head_params     = [p for n, p in model.named_parameters() if not n.startswith("backbone.") and "meta_" not in n and "learnable_slot_prior" not in n and "relation_head" not in n and "gamma" not in n and "gate" not in n and p.requires_grad]
        trainable_params = bb_params + fpn_params + stage3_params + head_params + meta_params + prior_params + gem_params + relation_params + gate_params

        param_groups = [
            {"params": bb_params,    "lr": CFG.lr_backbone, "weight_decay": CFG.weight_decay},
            {"params": head_params,  "lr": CFG.lr_head,     "weight_decay": CFG.weight_decay},
            {"params": prior_params, "lr": getattr(CFG, "lr_prior", 5.0e-3), "weight_decay": 0.0},
            {"params": gem_params,   "lr": getattr(CFG, "lr_gem", 1.0e-3),   "weight_decay": 0.0},
        ]
        if meta_params:
            param_groups.append({
                "params": meta_params, "lr": CFG.lr_head, "weight_decay": getattr(CFG, "meta_weight_decay", 0.03)
            })
        if gate_params:
            param_groups.append({
                "params": gate_params, "lr": getattr(CFG, "lr_gates", 2.0e-3), "weight_decay": 0.0
            })
        if fpn_params:
            param_groups.append({
                "params": fpn_params, "lr": getattr(CFG, "lr_fpn", CFG.lr_head), "weight_decay": CFG.weight_decay
            })
        elif stage3_params:
            param_groups.append({
                "params": stage3_params, "lr": getattr(CFG, "lr_stage3_proj", CFG.lr_head), "weight_decay": CFG.weight_decay
            })
        if relation_params:
            param_groups.append({
                "params": relation_params, "lr": getattr(CFG, "lr_relation", CFG.lr_head), "weight_decay": CFG.weight_decay
            })
        optimizer = torch.optim.AdamW(param_groups)

        total_steps = epochs * len(train_loader)
        max_lrs = [group["lr"] for group in optimizer.param_groups]
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=max_lrs,
            total_steps=total_steps,
            pct_start=CFG.pct_start
        )

        early_stopper = EarlyStopping(patience=CFG.early_stopping_patience, min_delta=CFG.early_stopping_min_delta)
        best_val_auc = 0.0
        scaler = torch.amp.GradScaler("cuda", enabled=(CFG.device.type == "cuda" and CFG.use_amp))
        top_k_checkpoints = []

        for epoch in range(1, epochs + 1):
            model.train()
            train_loss = 0.0
            optimizer.zero_grad(set_to_none=True)

            pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"S{stage_num}-F{fold} | Ep {epoch:02d}/{epochs:02d} [Train]", leave=False)

            for step, (imgs_u8, labels, weights, is_gold, c_flags, meta_t, is_right) in pbar:
                imgs_u8  = imgs_u8.to(CFG.device, non_blocking=True)
                labels   = labels.to(CFG.device, non_blocking=True)
                weights  = weights.to(CFG.device, non_blocking=True)
                is_gold  = is_gold.to(CFG.device, non_blocking=True)
                c_flags  = c_flags.to(CFG.device, non_blocking=True)
                meta_t   = meta_t.to(CFG.device, non_blocking=True)
                is_right = is_right.to(CFG.device, non_blocking=True)

                imgs, labels, weights, is_gold, c_dist, meta_t, is_mixed = gpu_augment_batch(
                    imgs_u8, labels, weights, is_gold, c_flags, meta_t, is_right=is_right
                )

                with torch.amp.autocast("cuda", enabled=(CFG.device.type == "cuda" and CFG.use_amp)):
                    if CFG.use_contrast_aux:
                        logits, aux_logits = model(imgs, contrast_flags=c_flags, meta_vec=meta_t)
                        main_loss = compute_loss(logits, labels, weights, is_gold, pu_weights=PU_WEIGHTS_TENSOR, fn_weights=FN_WEIGHTS_TENSOR)
                        log_probs = F.log_softmax(aux_logits.view(-1, CFG.n_contrast_types), dim=-1)
                        target_dist = c_dist.view(-1, CFG.n_contrast_types)
                        aux_loss = -(target_dist * log_probs).sum(dim=-1).mean()
                        loss = main_loss + CFG.aux_contrast_weight * aux_loss
                    else:
                        logits = model(imgs, contrast_flags=c_flags, meta_vec=meta_t)
                        loss = compute_loss(logits, labels, weights, is_gold, pu_weights=PU_WEIGHTS_TENSOR, fn_weights=FN_WEIGHTS_TENSOR)

                if torch.isnan(loss) or torch.isinf(loss):
                    continue

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)

                if not torch.isnan(grad_norm) and not torch.isinf(grad_norm):
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    ema.update(model)
                else:
                    scaler.update()
                optimizer.zero_grad(set_to_none=True)

                train_loss += loss.detach()
                if (step + 1) % 10 == 0 or (step + 1) == len(train_loader):
                    pbar.set_postfix({"loss": f"{loss.item():.4f}"})
                del imgs, labels, weights, is_gold, c_flags, meta_t, is_mixed, loss

            train_loss = (train_loss / len(train_loader)).item()

            # Validation Loop
            eval_model = ema.target(model)
            eval_model.eval()
            val_loss = 0.0
            val_preds, val_trues, val_wts = [], [], []

            with torch.inference_mode():
                for imgs_u8, labels, weights, is_gold, c_flags, meta_t, is_right in tqdm(val_loader, desc=f"S{stage_num}-F{fold} | Ep {epoch:02d}/{epochs:02d} [Val]", leave=False):
                    imgs_u8 = imgs_u8.to(CFG.device, non_blocking=True)
                    labels  = labels.to(CFG.device, non_blocking=True)
                    weights = weights.to(CFG.device, non_blocking=True)
                    is_gold_tensor = is_gold.to(CFG.device, non_blocking=True)
                    c_flags = c_flags.to(CFG.device, non_blocking=True)
                    meta_t  = meta_t.to(CFG.device, non_blocking=True)
                    is_right = is_right.to(CFG.device, non_blocking=True)

                    imgs = gpu_normalize_val_batch(imgs_u8, is_right=is_right)

                    with torch.amp.autocast("cuda", enabled=(CFG.device.type == "cuda" and CFG.use_amp)):
                        if CFG.use_contrast_aux:
                            out = eval_model(imgs, contrast_flags=c_flags, meta_vec=meta_t)
                            logits = out[0] if isinstance(out, tuple) else out
                            main_loss = compute_loss(logits, labels, weights, is_gold_tensor, pu_weights=PU_WEIGHTS_TENSOR, fn_weights=FN_WEIGHTS_TENSOR)
                            loss = main_loss
                        else:
                            logits = eval_model(imgs, contrast_flags=c_flags, meta_vec=meta_t)
                            loss = compute_loss(logits, labels, weights, is_gold_tensor, pu_weights=PU_WEIGHTS_TENSOR, fn_weights=FN_WEIGHTS_TENSOR)

                    val_loss += loss.item()
                    val_preds.append(torch.sigmoid(logits).cpu().numpy())
                    val_trues.append(labels.cpu().numpy())
                    val_wts.append(weights.cpu().numpy())
                    del imgs_u8, imgs, labels, weights, is_gold_tensor, c_flags, meta_t, is_right, logits, loss

            val_loss  = val_loss / len(val_loader)
            val_preds = np.concatenate(val_preds, axis=0)
            val_trues = np.concatenate(val_trues, axis=0)
            val_wts   = np.concatenate(val_wts,   axis=0)

            val_macro_auc = compute_macro_auc(val_trues, val_preds, weights=val_wts)

            # Monitor Diagnostics
            gem_p_val = model.backbone.pool.p.item() if hasattr(model.backbone, "pool") and hasattr(model.backbone.pool, "p") else 3.0
            fpn_g_val = model.backbone.fpn.gamma.item() if hasattr(model.backbone, "fpn") and hasattr(model.backbone.fpn, "gamma") else 0.0
            focal_g_val = model.head.focal_gamma.item() if hasattr(model.head, "focal_gamma") else 0.0

            # Gold Holdout Subset AUC
            gold_mask = val_df["is_gold"].values
            gold_str = ""
            if gold_mask.sum() > 0:
                gold_auc = compute_macro_auc(val_trues[gold_mask], val_preds[gold_mask], weights=val_wts[gold_mask])
                gold_str = f" | Gold Holdout AUC: {gold_auc:.4f} (n={gold_mask.sum()})"

            print(f"📊 Epoch {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro ROC-AUC: {val_macro_auc:.4f} | GeM p: {gem_p_val:.2f} | FPN γ: {fpn_g_val:.3f} | Focal g: {focal_g_val:.3f}{gold_str}")

            # Maintain Top-K Checkpoints for SWA
            sd_copy = {k: v.cpu().clone() for k, v in eval_model.state_dict().items()}
            top_k_checkpoints.append({"epoch": epoch, "auc": val_macro_auc, "state_dict": sd_copy})
            top_k_checkpoints = sorted(top_k_checkpoints, key=lambda x: x["auc"], reverse=True)[:CFG.top_k_swa]

            if val_macro_auc > best_val_auc:
                best_val_auc = val_macro_auc
                val_idx = stage_df[stage_df["fold"] == fold].index.to_numpy()
                oof_predictions[val_idx] = val_preds
                oof_targets[val_idx]     = val_trues
                oof_weights[val_idx]     = val_wts

                save_path = f"./checkpoints/{ckpt_prefix}_fold{fold}.pth"
                torch.save({
                    "epoch": epoch,
                    "stage": stage_num,
                    "model_state_dict": eval_model.state_dict(),
                    "val_macro_auc": best_val_auc,
                    "config": {
                        "backbone": CFG.backbone,
                        "head_type": CFG.head_type,
                        "feature_pool": CFG.feature_pool,
                        "img_size": CFG.img_size,
                        "num_slices": CFG.num_slices,
                        "num_slices_per_plane": CFG.num_slices_per_plane,
                        "use_contrast_embedding": CFG.use_contrast_embedding,
                        "use_metadata_injection": CFG.use_metadata_injection,
                        "use_meta_film": getattr(CFG, "use_meta_film", False),
                        "meta_dim": CFG.meta_dim,
                        "use_contrast_aux": CFG.use_contrast_aux,
                        "aux_contrast_weight": CFG.aux_contrast_weight,
                        "use_multi_prototype": getattr(CFG, "use_multi_prototype", False),
                        "num_prototypes": getattr(CFG, "num_prototypes", 1),
                        "use_dual_pooling": getattr(CFG, "use_dual_pooling", True),
                        "use_plane_decoupled_pooling": getattr(CFG, "use_plane_decoupled_pooling", True),
                        "focal_temperature": getattr(CFG, "focal_temperature", 2.0),
                        "broad_temperature": getattr(CFG, "broad_temperature", 2.0),
                        "use_multi_stage_fusion": getattr(CFG, "use_multi_stage_fusion", True),
                        "use_fpn_fusion": getattr(CFG, "use_fpn_fusion", True),
                        "use_finding_relation": getattr(CFG, "use_finding_relation", False),
                        "use_cross_plane": getattr(CFG, "use_cross_plane", False),
                        "use_gem_pooling": getattr(CFG, "use_gem_pooling", True),
                        "use_dynamic_contrast_gating": getattr(CFG, "use_dynamic_contrast_gating", False),
                        "dynamic_plane_gamma": getattr(CFG, "dynamic_plane_gamma", 1.0),
                        "target_cols": TARGET_COLS
                    }
                }, save_path)
                print(f"   💾 Saved Single Best -> {save_path} (AUC: {best_val_auc:.4f})")

            torch.cuda.empty_cache()
            gc.collect()

            if early_stopper.step(epoch, val_macro_auc):
                pass
            if early_stopper.early_stop:
                print(f"   🛑 Early Stopping triggered for Fold {fold} at Epoch {epoch}!")
                break

        # SWA Evaluation
        if len(top_k_checkpoints) > 1:
            top_eps = [t["epoch"] for t in top_k_checkpoints]
            print(f"\n⭐ Performing SWA over Top-{len(top_k_checkpoints)} Epochs: {top_eps}...")
            sds = [t["state_dict"] for t in top_k_checkpoints]
            avg_sd = {}
            for k in sds[0]:
                v0 = sds[0][k]
                if v0.is_floating_point():
                    avg_sd[k] = sum(sd[k].double() for sd in sds).div(len(sds)).to(v0.dtype)
                else:
                    avg_sd[k] = v0.clone()

            eval_model.load_state_dict({k: v.to(CFG.device) for k, v in avg_sd.items()})
            eval_model.eval()

            swa_preds = []
            with torch.inference_mode():
                for imgs_u8, labels, weights, is_gold, c_flags, meta_t, is_right in val_loader:
                    imgs_u8  = imgs_u8.to(CFG.device, non_blocking=True)
                    c_flags  = c_flags.to(CFG.device, non_blocking=True)
                    meta_t   = meta_t.to(CFG.device, non_blocking=True)
                    is_right = is_right.to(CFG.device, non_blocking=True)

                    imgs = gpu_normalize_val_batch(imgs_u8, is_right=is_right)
                    with torch.amp.autocast("cuda", enabled=(CFG.device.type == "cuda" and CFG.use_amp)):
                        out = eval_model(imgs, contrast_flags=c_flags, meta_vec=meta_t)
                        logits = out[0] if isinstance(out, tuple) else out
                    swa_preds.append(torch.sigmoid(logits).cpu().numpy())
                    del imgs_u8, imgs, c_flags, meta_t, logits, out
            swa_preds = np.concatenate(swa_preds, axis=0)
            swa_auc = compute_macro_auc(val_trues, swa_preds, val_wts)

            gold_mask = val_df["is_gold"].values
            gold_swa_str = ""
            if gold_mask.sum() > 0:
                gold_swa_auc = compute_macro_auc(val_trues[gold_mask], swa_preds[gold_mask], weights=val_wts[gold_mask])
                gold_swa_str = f" | SWA Gold Holdout AUC: {gold_swa_auc:.4f} (n={gold_mask.sum()})"

            print(f"🏆 SWA Model Val Macro ROC-AUC: {swa_auc:.4f}{gold_swa_str} vs Single-Best: {best_val_auc:.4f}")

            save_path = f"./checkpoints/{ckpt_prefix}_fold{fold}.pth"
            if swa_auc >= best_val_auc:
                print(f"   ✨ SWA OUTPERFORMED Single-Best by +{swa_auc - best_val_auc:.4f}! Overwriting checkpoint with SWA.")
                best_val_auc = swa_auc
                val_idx = stage_df[stage_df["fold"] == fold].index.to_numpy()
                oof_predictions[val_idx] = swa_preds

                torch.save({
                    "epoch": top_eps,
                    "stage": stage_num,
                    "model_state_dict": avg_sd,
                    "val_macro_auc": swa_auc,
                    "swa": True,
                    "config": {
                        "backbone": CFG.backbone,
                        "head_type": CFG.head_type,
                        "feature_pool": CFG.feature_pool,
                        "img_size": CFG.img_size,
                        "num_slices": CFG.num_slices,
                        "num_slices_per_plane": CFG.num_slices_per_plane,
                        "use_contrast_embedding": CFG.use_contrast_embedding,
                        "use_metadata_injection": CFG.use_metadata_injection,
                        "use_meta_film": getattr(CFG, "use_meta_film", False),
                        "meta_dim": CFG.meta_dim,
                        "use_contrast_aux": CFG.use_contrast_aux,
                        "aux_contrast_weight": CFG.aux_contrast_weight,
                        "use_multi_prototype": getattr(CFG, "use_multi_prototype", False),
                        "num_prototypes": getattr(CFG, "num_prototypes", 1),
                        "use_dual_pooling": getattr(CFG, "use_dual_pooling", True),
                        "use_plane_decoupled_pooling": getattr(CFG, "use_plane_decoupled_pooling", True),
                        "focal_temperature": getattr(CFG, "focal_temperature", 2.0),
                        "broad_temperature": getattr(CFG, "broad_temperature", 2.0),
                        "use_multi_stage_fusion": getattr(CFG, "use_multi_stage_fusion", True),
                        "use_fpn_fusion": getattr(CFG, "use_fpn_fusion", True),
                        "use_finding_relation": getattr(CFG, "use_finding_relation", False),
                        "use_cross_plane": getattr(CFG, "use_cross_plane", False),
                        "use_gem_pooling": getattr(CFG, "use_gem_pooling", True),
                        "use_dynamic_contrast_gating": getattr(CFG, "use_dynamic_contrast_gating", False),
                        "dynamic_plane_gamma": getattr(CFG, "dynamic_plane_gamma", 1.0),
                        "target_cols": TARGET_COLS
                    }
                }, save_path)
                print(f"   💾 Overwrote Single-Best with SWA -> {save_path} (AUC: {swa_auc:.4f})")
            else:
                print(f"   ℹ️ Single-Best ({best_val_auc:.4f}) >= SWA ({swa_auc:.4f}), preserving Single-Best -> {save_path}")

        fold_best_aucs.append(best_val_auc)
        print(f"🏆 Fold {fold} Final Macro ROC-AUC: {best_val_auc:.4f}\n")

        # Memory release between folds
        del model, optimizer, scheduler, train_loader, val_loader, train_ds, val_ds
        if 'eval_model' in locals():
            del eval_model
        if 'ema' in locals():
            del ema
        torch.cuda.empty_cache()
        gc.collect()

    return oof_predictions, oof_targets, oof_weights, fold_best_aucs


# ==============================================================================
# 📊 2. MODULAR OOF BENCHMARK EVALUATOR
# ==============================================================================
def evaluate_oof_benchmark(oof_predictions, oof_targets, oof_weights, df_eval, fold_best_aucs, stage_title="STAGE 1", save_csv_path="./checkpoints/oof_predictions.csv"):
    trained_mask = df_eval['fold'].isin(CFG.train_folds).values

    # Fold-wise Rank Normalization
    oof_predictions_ranked = oof_predictions.copy()
    for f in CFG.train_folds:
        f_mask = (df_eval['fold'].values == f)
        if f_mask.sum() > 0:
            for c_idx in range(len(TARGET_COLS)):
                preds_fold = oof_predictions[f_mask, c_idx]
                oof_predictions_ranked[f_mask, c_idx] = rankdata(preds_fold) / len(preds_fold)

    per_class_confident_aucs        = []
    per_class_confident_ranked_aucs = []
    per_class_full_aucs             = []
    per_class_full_ranked_aucs      = []
    per_class_gold_aucs             = []
    per_class_gold_ranked_aucs      = []
    class_stats = []

    gold_mask_arr = (df_eval['is_gold'].values == True) & trained_mask

    for i, col in enumerate(TARGET_COLS):
        t, p, p_rk, w = oof_targets[:, i], oof_predictions[:, i], oof_predictions_ranked[:, i], oof_weights[:, i]

        # Subset A: Confident (w > 0.3)
        valid_conf = trained_mask & (w > 0.3) & (~np.isnan(t))
        tv_c = (t[valid_conf] > 0.5).astype(np.int32)
        pv_c = p[valid_conf]
        pv_c_rk = p_rk[valid_conf]
        auc_conf = roc_auc_score(tv_c, pv_c) if len(np.unique(tv_c)) > 1 else np.nan
        auc_conf_rk = roc_auc_score(tv_c, pv_c_rk) if len(np.unique(tv_c)) > 1 else np.nan
        if not np.isnan(auc_conf):
            per_class_confident_aucs.append(auc_conf)
            per_class_confident_ranked_aucs.append(auc_conf_rk)

        # Subset B: Full Unweighted (All studies in trained folds)
        valid_full = trained_mask & (~np.isnan(t))
        tv_f = (t[valid_full] > 0.5).astype(np.int32)
        pv_f = p[valid_full]
        pv_f_rk = p_rk[valid_full]
        auc_full = roc_auc_score(tv_f, pv_f) if len(np.unique(tv_f)) > 1 else np.nan
        auc_full_rk = roc_auc_score(tv_f, pv_f_rk) if len(np.unique(tv_f)) > 1 else np.nan
        if not np.isnan(auc_full):
            per_class_full_aucs.append(auc_full)
            per_class_full_ranked_aucs.append(auc_full_rk)

        # Subset C: Gold Standard Cases
        valid_gold = gold_mask_arr & (~np.isnan(t))
        tv_g = (t[valid_gold] > 0.5).astype(np.int32)
        pv_g = p[valid_gold]
        pv_g_rk = p_rk[valid_gold]
        auc_gold = roc_auc_score(tv_g, pv_g) if len(np.unique(tv_g)) > 1 else np.nan
        auc_gold_rk = roc_auc_score(tv_g, pv_g_rk) if len(np.unique(tv_g)) > 1 else np.nan
        if not np.isnan(auc_gold):
            per_class_gold_aucs.append(auc_gold)
            per_class_gold_ranked_aucs.append(auc_gold_rk)

        class_stats.append({
            'target': col,
            'auc_conf': auc_conf, 'auc_conf_rk': auc_conf_rk, 'pos_conf': int(tv_c.sum()), 'tot_conf': len(tv_c),
            'auc_full': auc_full, 'auc_full_rk': auc_full_rk, 'pos_full': int(tv_f.sum()), 'tot_full': len(tv_f),
            'auc_gold': auc_gold, 'auc_gold_rk': auc_gold_rk, 'pos_gold': int(tv_g.sum()), 'tot_gold': len(tv_g),
        })

    print('\n' + '='*86)
    print(f'🏆 {stage_title} — OUT-OF-FOLD (OOF) BENCHMARK SUMMARY (RANK NORMALIZATION)')
    print('='*86)
    if len(fold_best_aucs) > 0:
        fold_str = ', '.join([f'F{f}:{a:.4f}' for f, a in zip(CFG.train_folds, fold_best_aucs)])
        print(f'⭐ 1. MEAN OF FOLDS MACRO AUC   : {np.mean(fold_best_aucs):.4f}  [{fold_str}]')
    if per_class_confident_aucs:
        print(f'⭐ 2. POOLED CONFIDENT OOF AUC   : {np.mean(per_class_confident_aucs):.4f}  ->  Rank-Norm: {np.mean(per_class_confident_ranked_aucs):.4f}  (w > 0.3 subset)')
    if per_class_full_aucs:
        print(f'⭐ 3. POOLED FULL ALL-DATA AUC   : {np.mean(per_class_full_aucs):.4f}  ->  Rank-Norm: {np.mean(per_class_full_ranked_aucs):.4f}  (All {trained_mask.sum():,} studies)')
    if per_class_gold_aucs:
        print(f'⭐ 4. OFFICIAL 58 GOLD STANDARDS : {np.mean(per_class_gold_aucs):.4f}  ->  Rank-Norm: {np.mean(per_class_gold_ranked_aucs):.4f}  ({gold_mask_arr.sum()} doctor cases)')
    print('='*86)

    print(f"\n{'Target Pathology':<18} | {'Confident (Raw -> Rank)':<25} | {'Full (Raw -> Rank)':<23} | {'Gold (Raw -> Rank)':<20}")
    print('-'*90)
    for s in class_stats:
        str_conf = f"{s['auc_conf']:.4f} -> {s['auc_conf_rk']:.4f}" if not np.isnan(s['auc_conf']) else "N/A"
        str_full = f"{s['auc_full']:.4f} -> {s['auc_full_rk']:.4f}" if not np.isnan(s['auc_full']) else "N/A"
        str_gold = f"{s['auc_gold']:.4f} -> {s['auc_gold_rk']:.4f}" if not np.isnan(s['auc_gold']) else "N/A"
        print(f"{s['target']:<18} | {str_conf:<25} | {str_full:<23} | {str_gold:<20}")
    print('='*90)

    id_cols = [c for c in ['StudyInstanceUID', 'study_id', 'series_id', 'fold', 'is_gold'] if c in df_eval.columns]
    oof_df = df_eval[id_cols].copy()
    for i, col in enumerate(TARGET_COLS):
        oof_df[f'pred_{col}'] = oof_predictions[:, i]
        oof_df[f'pred_ranked_{col}'] = oof_predictions_ranked[:, i]
        oof_df[f'target_{col}'] = oof_targets[:, i]
        oof_df[f'weight_{col}'] = oof_weights[:, i]
    os.makedirs(os.path.dirname(save_csv_path) if os.path.dirname(save_csv_path) else ".", exist_ok=True)
    oof_df.to_csv(save_csv_path, index=False)
    print(f"\n💾 Saved Out-Of-Fold Predictions to: {save_csv_path}")
    print('='*86 + '\n')
    return oof_df


# ==============================================================================
# 🧠 3. CROSS-MODAL KNOWLEDGE TRANSFER: SELECTIVE SELF-CORRECTION
# ==============================================================================
def run_cross_modal_self_correction(df, oof_preds, trained_folds, target_cols, conf_cols, cfg=CFG):
    """
    🔬 Thực hiện Self-Correction theo 4 chốt chặn kỹ thuật:
    1. Giữ nguyên 100% nhãn chuẩn 58 ca Gold Standard.
    2. Tập confident (w > 0.3): Soft blending y_new = α*y_llm + (1-α)*p_vis.
    3. Tập unmentioned/uncertain (w <= 0.3):
       - Nếu Visual Model tự tin (2|p_vis - 0.5| >= tau): Giải cứu Text thiếu!
       - Nếu cả Text lẫn Visual đều không chắc chắn: Hạ weight về 0.05 để chống nhiễu.
    """
    print("\n" + "="*86)
    print("🧠 RUNNING CROSS-MODAL SELECTIVE SELF-CORRECTION (TEXT -> VISION TRANSFER)")
    print("="*86)

    refined_df = df.copy()
    tau_vis = getattr(cfg, "self_correct_threshold", 0.40)
    w_uncertain = getattr(cfg, "self_correct_uncertain_w", 0.05)

    rescued_count = 0
    total_low_conf = 0

    for c_idx, (col, c_conf) in enumerate(zip(target_cols, conf_cols)):
        y_llm = df[col].values.copy()
        w_llm = df[c_conf].values.copy()
        p_vis = oof_preds[:, c_idx]
        is_gold = df['is_gold'].values

        conf_vis = np.clip(2.0 * np.abs(p_vis - 0.5), 0.0, 1.0)
        alpha = w_llm.copy()

        y_refined = y_llm.copy()
        w_refined = w_llm.copy()

        non_gold_mask = (~is_gold)
        low_conf_mask = non_gold_mask & (w_llm <= 0.30)
        high_conf_mask = non_gold_mask & (w_llm > 0.30)

        total_low_conf += low_conf_mask.sum()

        # A. High-Confidence: Soft Blending
        y_refined[high_conf_mask] = alpha[high_conf_mask] * y_llm[high_conf_mask] + (1.0 - alpha[high_conf_mask]) * p_vis[high_conf_mask]

        # B. Low-Confidence: Selective Relabeling
        rescue_mask = low_conf_mask & (conf_vis >= tau_vis)
        rescued_count += rescue_mask.sum()
        y_refined[rescue_mask] = alpha[rescue_mask] * y_llm[rescue_mask] + (1.0 - alpha[rescue_mask]) * p_vis[rescue_mask]
        w_refined[rescue_mask] = np.maximum(w_llm[rescue_mask], conf_vis[rescue_mask])

        # Ambig Mask: Giữ nhãn nhưng hạ trọng số về w_uncertain
        ambig_mask = low_conf_mask & (conf_vis < tau_vis)
        w_refined[ambig_mask] = w_uncertain

        # C. 58 ca Gold Standard: BẢO VỆ TUYỆT ĐỐI 100%
        y_refined[is_gold] = df.loc[is_gold, col].values
        w_refined[is_gold] = 1.0

        refined_df[col] = np.clip(y_refined, 0.01, 0.99)
        refined_df[c_conf] = np.clip(w_refined, 0.0, 1.0)
        # 🌟 Sync raw_ column so validation in Stage 2 evaluates against refined ground truth
        refined_df[f"raw_{col}"] = refined_df[col].values.copy()

    save_path = getattr(cfg, "refined_labels_save_path", "./checkpoints/llm_labels_v5_refined.csv")
    os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else ".", exist_ok=True)
    refined_df.to_csv(save_path, index=False)

    print(f"✅ Self-Correction Summary:")
    print(f"   • Tổng số vị trí nhãn Text mập mờ (w <= 0.30): {total_low_conf:,}")
    print(f"   • Số vị trí được Visual Model GIẢI CỨU thành công: {rescued_count:,} ({rescued_count / max(1, total_low_conf) * 100:.1f}%)")
    print(f"   • Số vị trí mập mờ cả hai bên (được hạ w={w_uncertain} chống nhiễu): {total_low_conf - rescued_count:,}")
    print(f"💾 File nhãn tinh chế đã sẵn sàng cho STAGE 2: {save_path}")
    print("="*86 + "\n")
    return refined_df


# ==============================================================================
# 🚀 4. EXECUTE STAGE 1: PURE VISUAL PRETRAINING (5 FOLDS)
# ==============================================================================
CFG.stage1_confident_only = True
print("="*86)
print("🚀 [STAGE 1] STARTING PURE VISUAL PRETRAINING (5 FOLDS)")
print(f"   🎯 Per-Finding Loss Masking: Gradient zeroed on ambiguous text labels (w <= {CFG.min_train_confidence})")
print(f"   🛡️ Gold Standards (58 cases): 100% strictly protected with {CFG.gold_weight}x sample weight")
print("="*86)

oof_preds_s1, oof_targets_s1, oof_weights_s1, fold_aucs_s1 = run_stage_training(
    stage_df=df,
    stage_num=1,
    epochs=getattr(CFG, "stage1_epochs", CFG.epochs),
    ckpt_prefix="best_convnext_tiny_kaggle_stage1"
)

oof_df_s1 = evaluate_oof_benchmark(
    oof_preds_s1, oof_targets_s1, oof_weights_s1, df, fold_aucs_s1,
    stage_title="STAGE 1 (PRE-REFINEMENT)",
    save_csv_path="./checkpoints/oof_predictions_stage1.csv"
)


🚀 [STAGE 1] STARTING PURE VISUAL PRETRAINING (5 FOLDS)
   🎯 Per-Finding Loss Masking: Gradient zeroed on ambiguous text labels (w <= 0.3)
   🛡️ Gold Standards (58 cases): 100% strictly protected with 3.0x sample weight

🚀 STARTING STAGE 1 TRAINING FOR FOLDS: [0, 1, 2, 3, 4] (11 Epochs OneCycleLR)

========================== STAGE 1 — FOLD 0 ==========================
🎯 [STAGE 1 ACTIVE] Fold 0 Train: Áp dụng Per-Finding Loss Masking (w <= 0.3).
   🛡️ Val fold 0 giữ nguyên 100% (882 ca) cho đánh giá khách quan.
⚖️ Fold 0 Auto-Calibrated PU Weights (Negative scale):
   {'ACL': 0.7, 'MCL': 0.7, 'Medial Meniscus': 0.624, 'Lateral Meniscus': 0.7, 'Medial OA': 0.686, 'Lateral OA': 0.694, 'PF OA': 0.668, 'Effusion': 0.662, 'Synovitis': 0.439, "Baker's": 0.587, 'Contusion': 0.674, 'Fracture': 0.697}
⚖️ Fold 0 Auto-Calibrated FN Weights (Positive boost):
   {'ACL': 1.266, 'MCL': 1.355, 'Medial Meniscus': 1.033, 'Lateral Meniscus': 1.279, 'Medial OA': 1.08, 'Lateral OA': 1.158, 'PF OA': 1.015, 'E

   ✅ Loaded pretrained timm/convnext_tiny.in12k_ft_in1k from timm hub.
⚡ Model Config: Dual-Pooling=True (Broad τ=2.0), FPN-Fusion=True, Slot-Prior=True


S1-F0 | Ep 01/11 [Train]:   0%|          | 0/220 [00:00<?, ?it/s]

S1-F0 | Ep 01/11 [Val]:   0%|          | 0/14 [00:00<?, ?it/s]

📊 Epoch 01/11 | Train Loss: 0.3026 | Val Loss: 0.2569 | Val Macro ROC-AUC: 0.7922 | GeM p: 3.00 | FPN γ: 0.041 | Focal g: 0.000 | Gold Holdout AUC: 0.8017 (n=12)
   💾 Saved Single Best -> ./checkpoints/best_convnext_tiny_kaggle_stage1_fold0.pth (AUC: 0.7922)


S1-F0 | Ep 02/11 [Train]:   0%|          | 0/220 [00:00<?, ?it/s]

... (139 outputs left out)

📊 Epoch 09/11 | Train Loss: 0.1432 | Val Loss: 0.1864 | Val Macro ROC-AUC: 0.8939 | GeM p: 3.00 | FPN γ: 0.019 | Focal g: 0.000 | Gold Holdout AUC: 0.9304 (n=11)
   💾 Saved Single Best -> ./checkpoints/best_convnext_tiny_kaggle_stage1_fold4.pth (AUC: 0.8939)
   ⏳ EarlyStopping: 1/2 (Best AUC: 0.8939)


S1-F4 | Ep 10/11 [Train]:   0%|          | 0/220 [00:00<?, ?it/s]

S1-F4 | Ep 10/11 [Val]:   0%|          | 0/14 [00:00<?, ?it/s]

📊 Epoch 10/11 | Train Loss: 0.1338 | Val Loss: 0.1888 | Val Macro ROC-AUC: 0.8932 | GeM p: 3.00 | FPN γ: 0.016 | Focal g: 0.000 | Gold Holdout AUC: 0.9253 (n=11)
   ⏳ EarlyStopping: 2/2 (Best AUC: 0.8939)
   🛑 Early Stopping triggered! Stopping training for this fold.
   🛑 Early Stopping triggered for Fold 4 at Epoch 10!

⭐ Performing SWA over Top-3 Epochs: [9, 8, 7]...
🏆 SWA Model Val Macro ROC-AUC: 0.8944 | SWA Gold Holdout AUC: 0.9271 (n=11) vs Single-Best: 0.8939
   ✨ SWA OUTPERFORMED Single-Best by +0.0005! Overwriting checkpoint with SWA.
   💾 Overwrote Single-Best with SWA -> ./checkpoints/best_convnext_tiny_kaggle_stage1_fold4.pth (AUC: 0.8944)
🏆 Fold 4 Final Macro ROC-AUC: 0.8944


🏆 STAGE 1 (PRE-REFINEMENT) — OUT-OF-FOLD (OOF) BENCHMARK SUMMARY (RANK NORMALIZATION)
⭐ 1. MEAN OF FOLDS MACRO AUC   : 0.8936  [F0:0.8954, F1:0.8921, F2:0.8953, F3:0.8906, F4:0.8944]
⭐ 2. POOLED CONFIDENT OOF AUC   : 0.8912  ->  Rank-Norm: 0.8920  (w > 0.3 subset)
⭐ 3. POOLED FULL ALL-DATA AUC   : 0

---
## 7. Cross-Modal Knowledge Transfer: Selective Self-Correction & Stage 2 Champion Retraining


In [9]:
# ==============================================================================
# 🧠 CROSS-MODAL SELECTIVE SELF-CORRECTION & STAGE 2 CHAMPION RETRAINING
# ==============================================================================
if getattr(CFG, "run_two_stage_pipeline", True) and len(CFG.train_folds) > 0:
    # ── 1. Knowledge Transfer: Self-Correction ────────────────────────────────
    if 'oof_preds_s1' in locals():
        refined_df = run_cross_modal_self_correction(
            df, oof_preds_s1, CFG.train_folds, TARGET_COLS, CONF_COLS, cfg=CFG
        )
    elif os.path.exists(getattr(CFG, "refined_labels_save_path", "./checkpoints/llm_labels_v5_refined.csv")):
        print(f"📂 Loading existing refined labels from: {CFG.refined_labels_save_path}")
        refined_df = pd.read_csv(CFG.refined_labels_save_path)
    else:
        raise RuntimeError("No Stage 1 OOF predictions or existing refined labels found!")

    # ── 2. Memory Purge between Stages ────────────────────────────────────────
    torch.cuda.empty_cache()
    gc.collect()

    # ── 3. Stage 2: Champion Retraining on Refined Ground Truth ───────────────
    CFG.stage1_confident_only = False  # Train on all 100% refined labels without masking
    print("="*86)
    print("🏆 [STAGE 2] STARTING CHAMPION RETRAINING ON REFINED DATASET (5 FOLDS)")
    print(f"   🎯 Loss Masking: OFF (All refined labels active with high visual confidence)")
    print(f"   💾 Final Models: Saving directly to ./checkpoints/best_convnext_tiny_kaggle_fold{{fold}}.pth (folds {CFG.train_folds})")
    print("="*86)

    oof_preds_s2, oof_targets_s2, oof_weights_s2, fold_aucs_s2 = run_stage_training(
        stage_df=refined_df,
        stage_num=2,
        epochs=getattr(CFG, "stage2_epochs", CFG.epochs),
        ckpt_prefix="best_convnext_tiny_kaggle"  # Target submission checkpoints!
    )

    print("\n" + "="*86)
    print("🏆 STAGE 2 OUT-OF-FOLD (OOF) FINAL CHAMPION EVALUATION")
    print("="*86)
    oof_df_s2 = evaluate_oof_benchmark(
        oof_preds_s2, oof_targets_s2, oof_weights_s2, refined_df, fold_aucs_s2,
        stage_title="STAGE 2 (CHAMPION REFINED)",
        save_csv_path="./checkpoints/oof_predictions_kaggle_5folds.csv"
    )
    # Also save standard oof_predictions.csv
    oof_df_s2.to_csv("./checkpoints/oof_predictions.csv", index=False)
    print("✨ ALL 2 STAGES COMPLETE! Final 5-fold models are saved and ready for Kaggle submission!")
else:
    print("ℹ️ Two-Stage pipeline is disabled or no folds trained. Stage 1 models are preserved.")



🧠 RUNNING CROSS-MODAL SELECTIVE SELF-CORRECTION (TEXT -> VISION TRANSFER)
✅ Self-Correction Summary:
   • Tổng số vị trí nhãn Text mập mờ (w <= 0.30): 2,612
   • Số vị trí được Visual Model GIẢI CỨU thành công: 1,766 (67.6%)
   • Số vị trí mập mờ cả hai bên (được hạ w=0.05 chống nhiễu): 846
💾 File nhãn tinh chế đã sẵn sàng cho STAGE 2: ./checkpoints/llm_labels_v5_refined.csv

🏆 [STAGE 2] STARTING CHAMPION RETRAINING ON REFINED DATASET (5 FOLDS)
   🎯 Loss Masking: OFF (All refined labels active with high visual confidence)
   💾 Final Models: Saving directly to ./checkpoints/best_convnext_tiny_kaggle_fold{fold}.pth (folds [0, 1, 2, 3, 4])

🚀 STARTING STAGE 2 TRAINING FOR FOLDS: [0, 1, 2, 3, 4] (11 Epochs OneCycleLR)

========================== STAGE 2 — FOLD 0 ==========================
🎯 [STAGE 2 ACTIVE] Fold 0 Train: Huấn luyện trên nhãn tinh chế (Refined Soft Labels), không áp dụng masking.
⚖️ Fold 0 Auto-Calibrated PU Weights (Negative scale):
   {'ACL': 0.7, 'MCL': 0.7, 'Medial Men

S2-F0 | Ep 01/11 [Train]:   0%|          | 0/220 [00:00<?, ?it/s]

S2-F0 | Ep 01/11 [Val]:   0%|          | 0/14 [00:00<?, ?it/s]

📊 Epoch 01/11 | Train Loss: 0.3067 | Val Loss: 0.2555 | Val Macro ROC-AUC: 0.8117 | GeM p: 3.00 | FPN γ: 0.041 | Focal g: 0.000 | Gold Holdout AUC: 0.7980 (n=12)
   💾 Saved Single Best -> ./checkpoints/best_convnext_tiny_kaggle_fold0.pth (AUC: 0.8117)


S2-F0 | Ep 02/11 [Train]:   0%|          | 0/220 [00:00<?, ?it/s]

S2-F0 | Ep 02/11 [Val]:   0%|          | 0/14 [00:00<?, ?it/s]

... (138 outputs left out)

📊 Epoch 08/11 | Train Loss: 0.1488 | Val Loss: 0.1717 | Val Macro ROC-AUC: 0.9224 | GeM p: 3.00 | FPN γ: 0.011 | Focal g: 0.000 | Gold Holdout AUC: 0.9416 (n=11)
   ⏳ EarlyStopping: 1/2 (Best AUC: 0.9225)


S2-F4 | Ep 09/11 [Train]:   0%|          | 0/220 [00:00<?, ?it/s]

S2-F4 | Ep 09/11 [Val]:   0%|          | 0/14 [00:00<?, ?it/s]

📊 Epoch 09/11 | Train Loss: 0.1461 | Val Loss: 0.1718 | Val Macro ROC-AUC: 0.9222 | GeM p: 3.00 | FPN γ: 0.007 | Focal g: 0.000 | Gold Holdout AUC: 0.9416 (n=11)
   ⏳ EarlyStopping: 2/2 (Best AUC: 0.9225)
   🛑 Early Stopping triggered! Stopping training for this fold.
   🛑 Early Stopping triggered for Fold 4 at Epoch 9!

⭐ Performing SWA over Top-3 Epochs: [7, 8, 9]...
🏆 SWA Model Val Macro ROC-AUC: 0.9228 | SWA Gold Holdout AUC: 0.9383 (n=11) vs Single-Best: 0.9225
   ✨ SWA OUTPERFORMED Single-Best by +0.0003! Overwriting checkpoint with SWA.
   💾 Overwrote Single-Best with SWA -> ./checkpoints/best_convnext_tiny_kaggle_fold4.pth (AUC: 0.9228)
🏆 Fold 4 Final Macro ROC-AUC: 0.9228


🏆 STAGE 2 OUT-OF-FOLD (OOF) FINAL CHAMPION EVALUATION

🏆 STAGE 2 (CHAMPION REFINED) — OUT-OF-FOLD (OOF) BENCHMARK SUMMARY (RANK NORMALIZATION)
⭐ 1. MEAN OF FOLDS MACRO AUC   : 0.9227  [F0:0.9231, F1:0.9238, F2:0.9221, F3:0.9219, F4:0.9228]
⭐ 2. POOLED CONFIDENT OOF AUC   : 0.9223  ->  Rank-Norm: 0.9226  (w 

---
## 8. Checkpoint Weight Diagnostic & Activity Audit
Kiểm tra độ dịch chuyển của các tham số cấu trúc ($\Delta = |w - w_0|$) với chuẩn hóa khởi tạo chính xác (`prior_scale` baseline $\approx -0.31$, `plane_emb` Gaussian noise audit, và safe prefix stripping).


In [10]:
import os
import torch
import torch.nn.functional as F
import numpy as np

# ── 1. Đường dẫn tới checkpoint ConvNeXt ──────────────────────────────────────
CKPT_PATH = "./checkpoints/best_convnext_tiny_kaggle_fold0.pth"

if not os.path.exists(CKPT_PATH):
    alt_paths = [
        "best_convnext_tiny_kaggle_fold0.pth",
        "/content/checkpoints/best_convnext_tiny_kaggle_fold0.pth",
        "/content/best_convnext_tiny_kaggle_fold0.pth",
        "./checkpoints/best_convnext_tiny_fold0.pth",
        "./checkpoints/best_convnext_tiny_colab_fold0_swa.pth"
    ]
    for p in alt_paths:
        if os.path.exists(p):
            CKPT_PATH = p
            break

print(f"🔍 Đang kiểm tra checkpoint ConvNeXt: {CKPT_PATH}")
ck = torch.load(CKPT_PATH, map_location="cpu")

# Bóc tách state_dict và xử lý prefix an toàn (chỉ gỡ wrapper ở đầu chuỗi, bảo tồn backbone.model...)
raw_sd = ck["model_state_dict"] if "model_state_dict" in ck else ck
sd = {}
for k, v in raw_sd.items():
    clean_k = k
    if clean_k.startswith("module."):
        clean_k = clean_k[len("module."):]
    if clean_k.startswith("model.backbone.") or clean_k.startswith("model.head.") or clean_k.startswith("model.plane_"):
        clean_k = clean_k[len("model."):]
    sd[clean_k] = v

TARGET_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", 
    "Medial OA", "Lateral OA", "PF OA", "Effusion", 
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

print("\n" + "=" * 100)
print(f"📊 KẾT QUẢ HUẤN LUYỆN CHECKPOINT CONVNEXT:")
print(f"   • Epoch: {ck.get('epoch', 'N/A')}")
val_auc_val = ck.get('val_macro_auc', ck.get('val_auc', None))
val_auc_str = f"{val_auc_val:.4f}" if isinstance(val_auc_val, (int, float)) else str(val_auc_val or 'N/A')
print(f"   • Val Macro ROC-AUC: {val_auc_str}")
if 'gold_auc' in ck and ck.get('gold_auc') is not None:
    gold_val = ck.get('gold_auc')
    gold_str = f"{gold_val:.4f}" if isinstance(gold_val, (int, float)) else str(gold_val)
    print(f"   • Gold Holdout AUC : {gold_str}")
print("=" * 100)

# ── 2. Bảng tổng hợp mức độ hoạt động (Activity Audit) ─────────────────────────
# Tính toán giá trị khởi tạo chính xác 100% theo code định nghĩa
prior_strength = 0.55
prior_init_param = float(np.log(np.exp(prior_strength) - 1.0))  # ≈ -0.3102 (softplus^{-1}(0.55))

INIT_SPECS = {
    "head.focal_gate":                       (0.0,    "scaler_zero"),
    "head.plane_gate":                       (0.0,    "scaler_zero"),
    "head.learnable_slot_prior.delta_mu":    (0.0,    "scaler_zero"),
    "head.learnable_slot_prior.delta_sigma": (0.0,    "scaler_zero"),
    "head.learnable_slot_prior.delta_gate":  (0.0,    "scaler_zero"),
    "head.prior_scale":                      (prior_init_param, "prior_scale"),
    "head.logit_scale":                      (2.7726, "scaler_const"), # ln(16.0)
    "plane_emb":                             (0.0200, "gaussian_noise"), # N(0, 0.02^2)
    "backbone.fpn.gamma":                    (0.0500, "scaler_const"), # FPN initial gamma
    "backbone.stage3_gamma":                 (0.0500, "scaler_const"), # Stage3 initial gamma
    "head.relation_head.gamma":              (0.0,    "scaler_zero"),
    "depth_block.gamma":                     (0.0,    "scaler_zero"),
}

print(f"\n{'TÊN THÀNH PHẦN (MODULE)':<40} | {'SHAPE':<12} | {'MEAN |x|':<9} | {'MAX |x|':<9} | {'ĐỘ DỊCH CHUYỂN Δ':<16} | {'ĐÁNH GIÁ'}")
print("-" * 120)

for k, (init_val, mode) in INIT_SPECS.items():
    if k in sd:
        v = sd[k].float()
        mean_abs = v.abs().mean().item()
        max_abs = v.abs().max().item()
        
        if mode == "prior_scale":
            # So sánh tham số raw với log(exp(0.55)-1) ≈ -0.3102
            delta = (v - init_val).abs().mean().item()
            eff_scale = F.softplus(v).mean().item()
            delta_str = f"{delta:.4f} (eff={eff_scale:.3f})"
            status = "🟢 HOẠT ĐỘNG TỐT" if delta >= 0.05 else "🔴 GẦN NHƯ CHẾT (Scale giữ nguyên ~0.55)"
            
        elif mode == "gaussian_noise":
            # plane_emb khởi tạo N(0, 0.02^2): std ban đầu = 0.0200, E[|x|] ≈ 0.0160
            curr_std = v.std().item()
            std_growth = curr_std - init_val
            delta = std_growth
            delta_str = f"std: {curr_std:.4f} (Δstd={std_growth:+.4f})"
            if curr_std >= 0.035:
                status = "🟢 HOẠT ĐỘNG TỐT (std phát triển vượt phân bố ngẫu nhiên)"
            elif curr_std >= 0.025:
                status = "🟡 HỌC NHẸ (std tăng nhẹ)"
            else:
                status = "🔴 CHƯA HỌC / NHIỄU (std ≈ 0.02 tương đương phân bố ngẫu nhiên)"
                
        else:
            delta = (v - init_val).abs().mean().item()
            delta_str = f"{delta:.4f}"
            if delta < 0.01:
                status = "🔴 GẦN NHƯ CHẾT (Chưa học gì)"
            elif delta < 0.04:
                status = "🟡 YẾU (Học rất ít)"
            else:
                status = "🟢 HOẠT ĐỘNG TỐT"
                
        print(f"{k:<40} | {str(list(v.shape)):<12} | {mean_abs:<9.4f} | {max_abs:<9.4f} | {delta_str:<16} | {status}")
    else:
        print(f"{k:<40} | {'KHÔNG CÓ TRONG CHECKPOINT (Đã tắt)':<50}")

# ── 3. Soi GeM Pooling Power p (Nếu có trong model) ───────────────────────────
print("\n" + "=" * 100)
print("🌊 KIỂM TRA GENERALIZED MEAN (GeM) POOLING POWER p (Khởi tạo = 3.00):")
print("=" * 100)
gem_keys = [k for k in sd.keys() if "pool" in k and k.endswith(".p")]
if gem_keys:
    for gk in gem_keys:
        p_val = sd[gk].float().item()
        delta_p = p_val - 3.0
        print(f"   • {gk:<40} : p = {p_val:.3f} (Δp = {delta_p:+.3f} so với 3.00) -> {'🟢 HỌC MẠNH' if abs(delta_p) > 0.05 else '🟡 ÍT THAY ĐỔI'}")
else:
    print("   ℹ️ Không tìm thấy tham số GeM p trong checkpoint (hoặc p cố định / GeM tắt)")

# ── 4. Soi chi tiết từng class cho các cổng quan trọng ───────────────────────
print("\n" + "=" * 100)
print("🔬 CHI TIẾT THEO TỪNG BỆNH LÝ (Dual-Scale Broad Gate & Gaussian Prior Residuals):")
print("=" * 100)
print(f"{'Pathology':<18} | {'Broad Gate g':<22} | {'Δ_mu (Vị trí)':<15} | {'Δ_sigma (Độ rộng)':<17} | {'Δ_gate (Mặt cắt)'}")
print("-" * 100)

has_focal = "head.focal_gate" in sd
has_prior = "head.learnable_slot_prior.delta_mu" in sd

if has_focal or has_prior:
    fg = sd["head.focal_gate"].squeeze().numpy() if has_focal else np.zeros(12)
    d_mu = sd["head.learnable_slot_prior.delta_mu"].numpy() if has_prior else np.zeros((12, 3))
    d_sig = sd["head.learnable_slot_prior.delta_sigma"].numpy() if has_prior else np.zeros((12, 3))
    d_gate = sd["head.learnable_slot_prior.delta_gate"].numpy() if has_prior else np.zeros((12, 3))
    
    for i, col in enumerate(TARGET_COLS):
        # 🌟 Display both raw logit gate and effective sigmoid blend weight: sig(g) in (0, 1)
        fg_val = f"{fg[i]:+.4f}" if has_focal else "N/A"
        mu_val = f"mean {np.abs(d_mu[i]).mean():.3f}" if has_prior else "N/A"
        sig_val = f"mean {np.abs(d_sig[i]).mean():.3f}" if has_prior else "N/A"
        gate_val = f"mean {np.abs(d_gate[i]).mean():.3f}" if has_prior else "N/A"
        print(f"{col:<18} | {fg_val:<22} | {mu_val:<15} | {sig_val:<17} | {gate_val}")

# ── 4b. Kiểm tra chi tiết 768 kênh của Channel-Wise FPN Gating ────────────────
if "backbone.fpn.gamma" in sd and sd["backbone.fpn.gamma"].numel() > 1:
    gamma_ch = sd["backbone.fpn.gamma"].squeeze().cpu().numpy()
    top_pos_idx = np.argsort(gamma_ch)[-5:][::-1]
    top_low_idx = np.argsort(gamma_ch)[:5]
    print(f"\n🔬 CHI TIẾT CHANNEL-WISE FPN GATING (768 KÊNH):")
    print(f"   • Phân phối: Mean = {gamma_ch.mean():.4f} | Std = {gamma_ch.std():.4f} | Min = {gamma_ch.min():.4f} | Max = {gamma_ch.max():.4f}")
    print(f"   • Top-5 kênh hấp thụ Stage 3 mạnh nhất: {', '.join([f'Ch#{i} ({gamma_ch[i]:+.4f})' for i in top_pos_idx])}")
    print(f"   • Top-5 kênh ức chế/giữ nguyên Stage 4 : {', '.join([f'Ch#{i} ({gamma_ch[i]:+.4f})' for i in top_low_idx])}")

# ── 5. Kiểm tra các nhánh bổ trợ (Metadata Injection & Contrast Head) ──────────
print("\n" + "=" * 100)
print("🧬 KIỂM TRA CÁC NHÁNH BỔ TRỢ (Metadata Injection & Contrast Head):")
print("=" * 100)
aux_keys = [
    ("meta_finding_bias.0.weight", "Metadata MLP Input Layer"),
    ("meta_finding_bias.4.weight", "Metadata Finding Output Bias"),
    ("contrast_head.1.weight",     "Contrast Aux Linear 1"),
    ("contrast_head.4.weight",     "Contrast Aux Output Head")
]
for k, label in aux_keys:
    if k in sd:
        v = sd[k].float()
        print(f"{label:<30} ({k}): mean|w|={v.abs().mean():.4f}, std={v.std():.4f}")
    else:
        print(f"{label:<30} ({k}): KHÔNG CÓ TRONG CHECKPOINT")

print("\n" + "=" * 100)


🔍 Đang kiểm tra checkpoint ConvNeXt: ./checkpoints/best_convnext_tiny_kaggle_fold0.pth

📊 KẾT QUẢ HUẤN LUYỆN CHECKPOINT CONVNEXT:
   • Epoch: [9, 8, 7]
   • Val Macro ROC-AUC: 0.9231

TÊN THÀNH PHẦN (MODULE)                  | SHAPE        | MEAN |x|  | MAX |x|   | ĐỘ DỊCH CHUYỂN Δ | ĐÁNH GIÁ
------------------------------------------------------------------------------------------------------------------------
head.focal_gate                          | [12, 1]      | 0.2101    | 0.5013    | 0.2101           | 🟢 HOẠT ĐỘNG TỐT
head.plane_gate                          | [12, 3]      | 0.0707    | 0.2325    | 0.0707           | 🟢 HOẠT ĐỘNG TỐT
head.learnable_slot_prior.delta_mu       | [12, 3]      | 0.2922    | 0.7771    | 0.2922           | 🟢 HOẠT ĐỘNG TỐT
head.learnable_slot_prior.delta_sigma    | [12, 3]      | 0.3296    | 0.7959    | 0.3296           | 🟢 HOẠT ĐỘNG TỐT
head.learnable_slot_prior.delta_gate     | [12, 3]      | 0.1900    | 0.5457    | 0.1900           | 🟢 HOẠT ĐỘNG TỐT
